# Importing va config

In [1]:
from __future__ import annotations

import time
import types
import platform
from pathlib import Path
from dataclasses import dataclass

import hmmlearn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from scipy.special import gammaln, logsumexp
from scipy.stats import multivariate_normal
from hmmlearn.hmm import GaussianHMM
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Tương đương Path(__file__).resolve().parents[1] của các script gốc (đặt trong scripts/).
# Notebook được giả định đặt trong notebooks/, cùng cấp với scripts/ và data/.
ROOT = Path.cwd().resolve().parents[0]
if not (ROOT / "data").exists():
    if (Path.cwd() / "data").exists():          # notebook chạy ngay tại ROOT của repo
        ROOT = Path.cwd().resolve()
    else:
        raise FileNotFoundError(
            f"Không tìm thấy thư mục 'data' tại {ROOT} hay {Path.cwd()}. "
            "Hãy sửa biến ROOT ở cell này cho đúng vị trí repo (cần data/interim/...)."
        )
print(f"ROOT = {ROOT}")

ROOT = D:\Academic\UET\ATCS\shock-detector


In [2]:
_C_RAW = ROOT / "data" / "raw"
_C_INTERIM = ROOT / "data" / "interim"
_C_REPORTS = ROOT / "reports"
_C_FIGS = _C_REPORTS / "figures"
_C_TABLES = _C_REPORTS / "tables"
_C_LOGS = _C_REPORTS / "logs"
_C_G1_DIR = _C_LOGS
_C_LABELS_FILE = "MILKY_Recovery_Labels_202603_202608.csv"

_C_HORIZON = 5 
_C_DEEP_M = 2 + _C_HORIZON 
_C_END_CUT_HOURS = 24

_C_BRAKE_RATES = (0.10, 0.20, 0.30)
_C_RULE_K = (3, 5, 7, 10)
_C_F1_FEATURES = ["logk", "log_gap", "accel", "log_since_r1"]
_C_SEED = 42

_C_EXPECTED_TOTALS = {"rows": 14045, "baskets": 6081, "max_Ri": 55}
_C_EXPECTED_QUALITY = {"EXACT_UNIQUE": 9035, "EXACT_AMBIGUOUS_FIRST": 2573,
                       "REPORT_SECOND_ONLY": 2437}
_C_EXPECTED_CONT = {1: (0.41, 0.08, 6081), 2: (0.50, 0.15, 2523), 3: (0.70, 0.23, 1272),
                    5: (0.79, 0.31, 632), 7: (0.80, 0.32, 372), 10: (0.79, 0.38, 197),
                    15: (0.81, 0.43, 74), 20: (0.94, 0.59, 32)}
_C_EXPECTED_MONTH = {"03": (67.5, 0.37, 0.78, 2.7, 5.0), "04": (54.9, 0.43, 0.79, 3.5, 5.7),
                     "05": (48.8, 0.44, 0.75, 2.6, 6.0), "06": (46.6, 0.41, 0.84, 3.8, 6.1),
                     "07": (31.7, 0.44, 0.84, 3.4, 7.8), "08": (43.7, 0.44, 0.77, 3.9, 6.7)}

C = types.SimpleNamespace(
    ROOT=ROOT, RAW=_C_RAW, INTERIM=_C_INTERIM, REPORTS=_C_REPORTS, FIGS=_C_FIGS,
    TABLES=_C_TABLES, LOGS=_C_LOGS, G1_DIR=_C_G1_DIR, LABELS_FILE=_C_LABELS_FILE,
    HORIZON=_C_HORIZON, DEEP_M=_C_DEEP_M, END_CUT_HOURS=_C_END_CUT_HOURS,
    BRAKE_RATES=_C_BRAKE_RATES, RULE_K=_C_RULE_K, F1_FEATURES=_C_F1_FEATURES, SEED=_C_SEED,
    EXPECTED_TOTALS=_C_EXPECTED_TOTALS, EXPECTED_QUALITY=_C_EXPECTED_QUALITY,
    EXPECTED_CONT=_C_EXPECTED_CONT, EXPECTED_MONTH=_C_EXPECTED_MONTH,
)
print("C.INTERIM =", C.INTERIM)

C.INTERIM = D:\Academic\UET\ATCS\shock-detector\data\interim


# Đọc, làm sạch, kiểm tra chất lượng M1, và ghép dữ liệu nến đã đóng vào các event

In [3]:
ONE_MIN = pd.Timedelta(minutes=1)
LONG_GAP_MIN = 30
MIN_BARS_FULL_DAY = 600

_MT5_COLS = {"<DATE>": "date", "<TIME>": "clock", "<OPEN>": "open", "<HIGH>": "high",
             "<LOW>": "low", "<CLOSE>": "close", "<TICKVOL>": "tick_volume",
             "<VOL>": "real_volume", "<SPREAD>": "spread"}


def _detect_sep(path: Path) -> str:
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        line = f.readline()
    for s in ("\t", ";", ","):
        if s in line:
            return s
    raise ValueError(f"Không nhận ra dấu phân cách trong dòng đầu: {line!r}")


def read_mt5_m1(path) -> pd.DataFrame:
    path = Path(path)
    df = pd.read_csv(path, sep=_detect_sep(path), encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    missing = [c for c in _MT5_COLS if c not in df.columns and c != "<VOL>"]
    if missing:
        raise ValueError(f"Thiếu cột MT5: {missing}; có: {list(df.columns)}")
    df = df.rename(columns=_MT5_COLS)
    df["time"] = pd.to_datetime(df["date"].astype(str) + " " + df["clock"].astype(str),
                                format="%Y.%m.%d %H:%M:%S").astype("datetime64[ns]")
    keep = ["time", "open", "high", "low", "close", "tick_volume", "spread"]
    return df[keep]


def clean_m1(raw: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    st = {"n_raw": len(raw)}
    df = raw.copy()
    bad = df[["open", "high", "low", "close"]].isna().any(axis=1) | (df[["open", "high", "low", "close"]] <= 0).any(axis=1)
    st["n_nan_or_nonpos"] = int(bad.sum())
    df = df[~bad]
    st["n_not_whole_minute"] = int((df["time"].dt.second != 0).sum())
    st["n_dup_exact"] = int(df.duplicated().sum())                     # trùng hoàn toàn
    df = df.drop_duplicates()
    dup_t = df["time"].duplicated(keep=False)
    st["n_dup_conflict_times"] = int(df.loc[dup_t, "time"].nunique())  # cùng giờ, khác giá trị
    df = df.sort_values("time", kind="stable").drop_duplicates("time", keep="last")
    st["n_unsorted_input"] = int((raw["time"].diff() < pd.Timedelta(0)).sum())
    ohlc_bad = (df["high"] < df[["open", "close"]].max(axis=1)) | (df["low"] > df[["open", "close"]].min(axis=1))
    st["n_ohlc_inconsistent"] = int(ohlc_bad.sum())                    # chỉ đếm, giữ nguyên
    df = df.reset_index(drop=True)
    df["gap_min"] = df["time"].diff().dt.total_seconds().div(60)       # phút từ nến trước (1 = liên tục)
    st["n_clean"] = len(df)
    st["start"], st["end"] = df["time"].iloc[0], df["time"].iloc[-1]
    return df, st


def save_m1(df: pd.DataFrame, path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)


def load_m1_clean(path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    df["time"] = df["time"].astype("datetime64[ns]")
    return df


def classify_gaps(df: pd.DataFrame, long_gap_min: int = LONG_GAP_MIN) -> pd.DataFrame:
    """Liệt kê mọi khoảng trống (gap > 1 phút) và phân loại:
    weekend  : qua thứ 7/CN (đóng thứ 6 -> mở CN/T2)
    long     : >= long_gap_min phút thiếu, không qua cuối tuần (nghỉ hằng ngày, lễ)
    intraday : còn lại = phút thiếu trong giờ giao dịch
    """
    t = df["time"]
    g = df.loc[df["gap_min"] > 1, ["time", "gap_min"]].copy()
    g["prev_time"] = t.shift(1).loc[g.index]
    g["n_missing"] = (g["gap_min"] - 1).astype(int)
    prev_d, next_d = g["prev_time"].dt.normalize(), g["time"].dt.normalize()
    span_days = (next_d - prev_d).dt.days
    # qua cuối tuần nếu khoảng [prev, next] chứa thứ 7 (weekday 5)
    wd_prev = g["prev_time"].dt.weekday
    has_sat = ((5 - wd_prev) % 7) <= span_days
    g["kind"] = np.where(has_sat & (span_days >= 1), "weekend",
                         np.where(g["n_missing"] >= long_gap_min, "long", "intraday"))
    return g.reset_index(drop=True)


def quality_summary(df: pd.DataFrame, gaps: pd.DataFrame) -> pd.DataFrame:
    """Theo tháng: số nến, phút thiếu trong giờ giao dịch (intraday), tỷ lệ thiếu, số gap dài/cuối tuần."""
    m = df["time"].dt.to_period("M").astype(str).rename("month")
    out = df.groupby(m).size().rename("n_bars").to_frame()
    gm = gaps["time"].dt.to_period("M").astype(str)
    intra = gaps[gaps["kind"] == "intraday"]
    out["missing_intraday"] = intra.groupby(gm[intra.index])["n_missing"].sum()
    out["n_intraday_gaps"] = intra.groupby(gm[intra.index]).size()
    for kind in ("long", "weekend"):
        sub = gaps[gaps["kind"] == kind]
        out[f"n_{kind}_gaps"] = sub.groupby(gm[sub.index]).size()
    out = out.fillna(0).astype(int)
    out["missing_pct"] = 100 * out["missing_intraday"] / (out["n_bars"] + out["missing_intraday"])
    tot = out.sum(numeric_only=True)
    tot["missing_pct"] = 100 * tot["missing_intraday"] / (tot["n_bars"] + tot["missing_intraday"])
    out.loc["TOTAL"] = tot
    ic = [c for c in out.columns if c != "missing_pct"]
    out[ic] = out[ic].astype(int)
    return out


def hour_coverage(df: pd.DataFrame) -> pd.DataFrame:
    """Tháng x giờ server: tỷ lệ phút có nến, tính trên các ngày T2-T6 có dữ liệu."""
    d = df[df["time"].dt.weekday < 5]
    days = d.groupby(d["time"].dt.to_period("M").astype(str))["time"].apply(lambda s: s.dt.normalize().nunique())
    cnt = d.groupby([d["time"].dt.to_period("M").astype(str), d["time"].dt.hour]).size().unstack(fill_value=0)
    cnt = cnt.reindex(columns=range(24), fill_value=0)
    return cnt.div(days * 60, axis=0)


def spread_summary(df: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """Phân bố và trung vị theo giờ."""
    s = df["spread"]
    q = s.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).rename(lambda x: f"q{int(x*100):02d}")
    q["mean"], q["max"], q["frac_zero"] = s.mean(), s.max(), (s == 0).mean()
    by_hour = df.groupby(df["time"].dt.hour)["spread"].median()
    return q, by_hour


def decision_bar_key(t: pd.Series, offset=pd.Timedelta(0)) -> pd.Series:
    """Giờ mở của nến M1 mới nhất ĐÃ ĐÓNG tại thời điểm t theo M1.
    = floor(t + offset, phút) - 1 phút.  Nến floor(t) đang chạy -> loại."""
    return (pd.to_datetime(t).astype("datetime64[ns]") + offset).dt.floor("min") - ONE_MIN


def attach_last_closed(events: pd.DataFrame, m1: pd.DataFrame, time_col: str = "timestamp",
                       cols=("close",), offset=pd.Timedelta(0), prefix: str = "m1_") -> pd.DataFrame:
    """Gắn vào mỗi sự kiện giá trị của nến M1 mới nhất đã đóng trước t.

    m1: DataFrame có cột `time` (giờ mở nến) + các cột cần gắn (giá hoặc đặc trưng
        nhân quả đã tính sẵn trên M1).
    Trả về bản sao `events` (giữ thứ tự) thêm:
      bar_time : giờ mở nến được dùng (luôn <= floor(t+offset) - 1 phút)
      pos      : vị trí dòng của nến đó trong m1 (để tính cửa sổ lùi về quá khứ)
      bar_age_min : số phút từ nến đó đến nến lẽ ra phải dùng
                            (0 = liền trước; >0 = đang trong khoảng trống, ví dụ sau cuối tuần)
      <col>    : giá trị cột
    """
    m = m1[["time", *cols]].copy()
    m["time"] = m["time"].astype("datetime64[ns]")
    m["_pos"] = np.arange(len(m))
    if not m["time"].is_monotonic_increasing:
        raise ValueError("m1 phải sắp xếp tăng dần theo time")
    ev = events.copy()
    ev["_key"] = decision_bar_key(ev[time_col], offset)
    ev["_order"] = np.arange(len(ev))
    # backward + allow_exact_matches: lấy nến có giờ mở <= key, KHÔNG BAO GIỜ lấy nến sau key
    j = pd.merge_asof(ev.sort_values("_key")[["_key", "_order"]], m.rename(columns={"time": "_bar"}),
                      left_on="_key", right_on="_bar", direction="backward", allow_exact_matches=True)
    j = j.sort_values("_order").set_index(ev.index)
    ev[f"{prefix}bar_time"] = j["_bar"]
    ev[f"{prefix}pos"] = j["_pos"].astype("Int64")
    ev[f"{prefix}bar_age_min"] = (j["_key"] - j["_bar"]).dt.total_seconds().div(60)
    for c in cols:
        ev[f"{prefix}{c}"] = j[c]
    return ev.drop(columns=["_key", "_order"])

# Cài đặt BOCPD

In [4]:
LAM_DEFAULT = 120 
R_MAX_DEFAULT = 600 
K_CP_DEFAULT = 10 


@dataclass(frozen=True)
class NIGPrior:
    """Prior Normal-Inverse-Gamma: sigma^2 ~ IG(alpha0, beta0), mu | sigma^2 ~ N(mu0, sigma^2/kappa0)."""
    mu0: float = 0.0
    kappa0: float = 1.0
    alpha0: float = 2.0
    beta0: float = 1.0

    @classmethod
    def from_train(cls, x_train, kappa0: float = 1.0, alpha0: float = 2.0) -> "NIGPrior":
        """Prior từ dữ liệu TRAIN. Với alpha0 = 2: E[sigma^2] = beta0/(alpha0-1) = var(train)."""
        x = np.asarray(x_train, dtype=float)
        x = x[np.isfinite(x)]
        return cls(mu0=float(x.mean()), kappa0=kappa0, alpha0=alpha0,
                   beta0=float(x.var()) * (alpha0 - 1.0))


def _student_t_logpdf(x: float, mu, kappa, alpha, beta):
    """log predictive Student-t của NIG: df = 2a, loc = mu, scale^2 = b(k+1)/(a k)."""
    df = 2.0 * alpha
    scale2 = beta * (kappa + 1.0) / (alpha * kappa)
    z2 = (x - mu) ** 2 / scale2
    return (gammaln((df + 1) / 2) - gammaln(df / 2)
            - 0.5 * np.log(np.pi * df * scale2)
            - (df + 1) / 2 * np.log1p(z2 / df))


class BOCPD:
    """BOCPD online. Gọi `update(x_t)` lần lượt; mỗi lần trả đặc trưng tại t."""

    def __init__(self, prior: NIGPrior = NIGPrior(), lam: float = LAM_DEFAULT,
                 r_max: int = R_MAX_DEFAULT, k_cp: int = K_CP_DEFAULT):
        self.prior, self.r_max, self.k_cp = prior, int(r_max), int(k_cp)
        self.log_h = np.log(1.0 / lam)
        self.log_1mh = np.log1p(-1.0 / lam)
        # Khởi tạo (bước 1): P(r_0 = 0) = 1, tham số = prior.
        self.log_r = np.array([0.0])
        self.mu = np.array([prior.mu0]); self.kappa = np.array([prior.kappa0])
        self.alpha = np.array([prior.alpha0]); self.beta = np.array([prior.beta0])

    def update(self, x: float) -> dict:
        x = float(x)
        if not np.isfinite(x):
            raise ValueError("x không hữu hạn: xử lý NaN/gap trước khi đưa vào BOCPD")
        # Bước 3: predictive pi_t^(r) cho x_t theo từng run length của t-1.
        log_pi = _student_t_logpdf(x, self.mu, self.kappa, self.alpha, self.beta)
        # Bước 4 (growth) và bước 5 (changepoint).
        log_growth = self.log_r + log_pi + self.log_1mh
        log_cp = logsumexp(self.log_r + log_pi + self.log_h)
        new_log_r = np.concatenate(([log_cp], log_growth))
        # Bước 6-7: evidence và chuẩn hóa -> P(r_t | x_1..t).
        new_log_r -= logsumexp(new_log_r)
        # Bước 8: cập nhật sufficient statistics (eq. 12-13 dạng NIG).
        p = self.prior
        mu_n = (self.kappa * self.mu + x) / (self.kappa + 1.0)
        beta_n = self.beta + self.kappa * (x - self.mu) ** 2 / (2.0 * (self.kappa + 1.0))
        self.mu = np.concatenate(([p.mu0], mu_n))
        self.kappa = np.concatenate(([p.kappa0], self.kappa + 1.0))
        self.alpha = np.concatenate(([p.alpha0], self.alpha + 0.5))
        self.beta = np.concatenate(([p.beta0], beta_n))
        # Cắt run length dài nhất rồi chuẩn hóa lại (mục 2.4).
        if new_log_r.size > self.r_max + 1:
            new_log_r = new_log_r[: self.r_max + 1]
            new_log_r -= logsumexp(new_log_r)
            for name in ("mu", "kappa", "alpha", "beta"):
                setattr(self, name, getattr(self, name)[: self.r_max + 1])
        self.log_r = new_log_r

        prob = np.exp(new_log_r)
        r = np.arange(prob.size)
        return {
            "p_r0": float(prob[0]),                        # chẩn đoán, = H
            "exp_run": float((r * prob).sum()),            # E[r_t | x_1..t]
            "map_run": int(prob.argmax()),
            "cp_recent": float(prob[: self.k_cp + 1].sum()),  # ĐẠI DIỆN: P(r_t <= k_cp)
        }


def run_bocpd(x, prior: NIGPrior = NIGPrior(), lam: float = LAM_DEFAULT,
              r_max: int = R_MAX_DEFAULT, k_cp: int = K_CP_DEFAULT,
              return_matrix: bool = False):
    """Chạy BOCPD tuần tự trên cả chuỗi. Dòng t chỉ phụ thuộc x[:t+1].

    return_matrix=True trả thêm ma trận P(r_t | x_1..t) (T x (r_max+1)) để vẽ.
    """
    model = BOCPD(prior, lam, r_max, k_cp)
    x = np.asarray(x, dtype=float)
    rows, mat = [], (np.zeros((x.size, r_max + 1)) if return_matrix else None)
    for t, xt in enumerate(x):
        rows.append(model.update(xt))
        if return_matrix:
            p = np.exp(model.log_r)
            mat[t, : p.size] = p
    out = pd.DataFrame(rows)
    return (out, mat) if return_matrix else out

# Triển khai HMM nhận diện trạng thái, tính xác suất thị trường đang ở trạng thái theo từng mốc tgian

In [5]:
K_DEFAULT = 3


def _as_2d(X) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    return X[:, None] if X.ndim == 1 else X


def fit_hmm(X_train, n_states: int = K_DEFAULT, lengths=None, n_init: int = 5,
            seed: int = 0, n_iter: int = 200, min_covar: float = 1e-3) -> GaussianHMM:
    """Fit GaussianHMM trên TRAIN, n_init lần khởi tạo, giữ log-likelihood cao nhất."""
    X = _as_2d(X_train)
    best, best_ll = None, -np.inf
    for i in range(n_init):
        m = GaussianHMM(n_components=n_states, covariance_type="full", n_iter=n_iter,
                        min_covar=min_covar, random_state=seed + i, tol=1e-4)
        m.fit(X, lengths)
        ll = m.score(X, lengths)   # score = log P(X) qua forward: hợp lệ để chọn mô hình
        if ll > best_ll:
            best, best_ll = m, ll
    return best


def forward_filter(model: GaussianHMM, X, lengths=None, return_loglik: bool = False):
    """Filtered probability P(z_t | x_1..t), shape (T, K). Thứ tự state như model.

    lengths: các chuỗi con độc lập (ví dụ tách theo tuần); mỗi chuỗi bắt đầu
    lại từ startprob_. Đây là thuật toán forward thuần, không có backward pass.
    """
    X = _as_2d(X)
    T, K = X.shape[0], model.n_components
    log_pi = np.log(np.clip(model.startprob_, 1e-300, None))
    log_A = np.log(np.clip(model.transmat_, 1e-300, None))
    covars = model.covars_   # hmmlearn trả dạng full (K, d, d)
    log_b = np.column_stack([
        multivariate_normal.logpdf(X, mean=model.means_[k], cov=covars[k], allow_singular=False)
        .reshape(T) for k in range(K)])

    lengths = [T] if lengths is None else list(lengths)
    assert sum(lengths) == T, "sum(lengths) phải bằng số dòng X"
    log_alpha = np.empty((T, K))
    loglik, start = 0.0, 0
    for n in lengths:
        prev = None
        for t in range(start, start + n):
            if prev is None:
                la = log_pi + log_b[t]
            else:
                # alpha_t(j) propto [sum_i alpha_{t-1}(i) a_ij] * b_j(x_t) -- chỉ dùng quá khứ
                la = logsumexp(prev[:, None] + log_A, axis=0) + log_b[t]
            c = logsumexp(la)          # log hệ số scaling c_t (Rabiner eq. 91-93)
            loglik += c
            prev = la - c              # chuẩn hóa -> log P(z_t | x_1..t)
            log_alpha[t] = prev
        start += n
    probs = np.exp(log_alpha)
    return (probs, loglik) if return_loglik else probs


def map_states(model: GaussianHMM, vol_col: int = -1) -> tuple[np.ndarray, pd.DataFrame]:
    """Map state -> nhãn theo variance (mean của cột log vol), tăng dần.

    Trả `order`: order[j] = chỉ số state gốc của nhãn j (0 = vol thấp nhất,
    K-1 = vol cao nhất), và bảng mô tả (mean, var, thời gian ở state 1/(1-a_ii)).
    Dùng: probs_mapped = probs[:, order].
    """
    X_means = model.means_
    d = X_means.shape[1]
    if d == 1:   # 1 chiều: sắp theo variance
        key = model.covars_[:, 0, 0]
    else:        # nhiều chiều: sắp theo mean của cột log vol
        key = X_means[:, vol_col]
    order = np.argsort(key)
    K = model.n_components
    names = {2: ["low", "high"], 3: ["low", "mid", "high"]}.get(K, [f"s{j}" for j in range(K)])
    rows = []
    for j, s in enumerate(order):
        row = {"label": names[j], "orig_state": int(s),
               "expected_duration": 1.0 / max(1.0 - model.transmat_[s, s], 1e-12)}
        for c in range(d):
            row[f"mean_{c}"] = X_means[s, c]
            row[f"var_{c}"] = model.covars_[s, c, c]
        rows.append(row)
    return order, pd.DataFrame(rows)


def regime_feature(model: GaussianHMM, X, lengths=None, vol_col: int = -1) -> np.ndarray:
    """ĐẠI DIỆN họ 5: P(state vol cao nhất | x_1..t) -- filtered, đã map nhãn."""
    order, _ = map_states(model, vol_col)
    return forward_filter(model, X, lengths)[:, order[-1]]

# Đánh giá mô hình, mô phỏng chiến lược phanh

In [6]:
def make_fold(samples, month):
    """Train = dòng của các tháng trước, loại basket kết thúc sau mốc cắt (purge).
    Test = dòng thuộc tháng `month`."""
    ms = pd.Timestamp(month + "-01")
    train = samples[samples["t_end"] < ms]
    test = samples[samples["month"] == month]
    n_purged = int(((samples["timestamp"] < ms) & (samples["t_end"] >= ms)).sum())
    return train, test, n_purged


def walk_forward(samples, test_months, features=None, rates=C.BRAKE_RATES, C_reg=1.0):
    """features=None -> luật theo độ sâu (score = k). Ngược lại logistic L2.
    Ngưỡng cho từng tỷ lệ phanh được chọn trên TRAIN, không dùng test."""
    outs = []
    for m in test_months:
        tr, te, _ = make_fold(samples, m)
        if len(te) == 0 or len(tr) == 0:
            continue
        if features is None:
            s_tr, s_te = tr["k"].to_numpy(float), te["k"].to_numpy(float)
        else:
            pipe = make_pipeline(StandardScaler(), LogisticRegression(C=C_reg, max_iter=1000))
            pipe.fit(tr[features], tr["y"])
            s_tr = pipe.predict_proba(tr[features])[:, 1]
            s_te = pipe.predict_proba(te[features])[:, 1]
        o = te[["basket_id", "k", "y", "M", "month", "day"]].copy()
        o["score"] = s_te
        for q in rates:
            o[f"thr_{round(q * 100)}"] = np.quantile(s_tr, 1 - q)
        o["fold"] = m
        outs.append(o)
    return pd.concat(outs, ignore_index=True)


def pr_auc(y, s):
    y = np.asarray(y)
    return float("nan") if y.sum() == 0 else float(average_precision_score(y, s))


def paired_bootstrap(y, s_a, s_b, groups, n_boot=1000, seed=C.SEED):
    """Bootstrap theo cụm (ngày), ghép cặp hai mô hình trên cùng mẫu lặp."""
    y, s_a, s_b = map(np.asarray, (y, s_a, s_b))
    codes, uniq = pd.factorize(np.asarray(groups))
    idx_by = [np.flatnonzero(codes == i) for i in range(len(uniq))]
    rng = np.random.default_rng(seed)
    a, b = [], []
    for _ in range(n_boot):
        pick = rng.integers(0, len(uniq), len(uniq))
        idx = np.concatenate([idx_by[i] for i in pick])
        if y[idx].sum() == 0:
            continue
        a.append(average_precision_score(y[idx], s_a[idx]))
        b.append(average_precision_score(y[idx], s_b[idx]))
    a, b = np.array(a), np.array(b)
    ci = lambda x: (float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5)))
    return dict(ap_a=pr_auc(y, s_a), ci_a=ci(a), ap_b=pr_auc(y, s_b), ci_b=ci(b),
                delta=pr_auc(y, s_a) - pr_auc(y, s_b), ci_delta=ci(a - b),
                frac_delta_pos=float((a - b > 0).mean()), n_boot=len(a))


def brake_sim(oos, thr):
    """HALT tại dòng đầu tiên có score >= thr trong mỗi basket (tuần tự).
    avoided = M - k + 1: số bậc Rk..RM không được mở. Đây là CẬN TRÊN, không phải giảm lỗ.
    'Bắt kịp' = HALT đầu tiên rơi vào dòng y = 1 (còn >= 5 bậc nữa)."""
    d = oos[["basket_id", "k", "y", "M", "score"]].copy()
    d["flag"] = d["score"].to_numpy() >= np.asarray(thr)
    b = d.groupby("basket_id", sort=False).agg(M=("M", "first"))
    first = (d[d["flag"]].sort_values(["basket_id", "k"]).groupby("basket_id", sort=False)
             .first()[["k", "y"]].rename(columns={"k": "k_halt", "y": "y_halt"}))
    b = b.join(first)
    halted = b["k_halt"].notna()
    deep = b["M"] >= C.DEEP_M
    avoided = (b["M"] - b["k_halt"] + 1).where(halted, 0)
    n_b, n_deep = len(b), int(deep.sum())
    caught = int(((b["y_halt"] == 1) & deep).sum())
    false_h = int((b["y_halt"] == 0).sum())
    return dict(
        brake_rate_rows=float(d["flag"].mean()),
        n_baskets=n_b, n_deep=n_deep,
        halted_pct=float(halted.mean()),
        deep_caught=caught,
        deep_caught_pct=caught / n_deep if n_deep else float("nan"),
        avoided_total=int(avoided.sum()),
        avoided_pct=float(avoided.sum() / max((b["M"] - 1).sum(), 1)),
        avoided_deep=int(avoided[deep].sum()),
        false_flag_rows=int((d["flag"] & (d["y"] == 0)).sum()),
        false_halts=false_h,
        false_halts_per100=100 * false_h / n_b,
    )

# Tạo đặc trưng từ nến M1, gắn vào từng event, kiểm tra xem có bị lộ future data

Future data ở đây là các nến chưa đóng/chưa hình thành

In [7]:
RV_WIN = 30
TREND_WIN = 60
TREND_WIN_SHORT = 30
BAR_COLS = ("lrv30", "ser60", "ser30", "ret60_z")
MAIN_FEATURES = {"vol": ["vol_z"], "trend": ["ser60"]}


def _contig_run(gap_min: np.ndarray) -> np.ndarray:
    """Số return 1 phút liên tục kết thúc tại mỗi nến (0 ở nến đầu / ngay sau khoảng trống)."""
    ok = np.nan_to_num(gap_min, nan=0.0) == 1
    run = np.zeros(len(ok), dtype=np.int64)
    c = 0
    for i, v in enumerate(ok):          # 200k nến, đủ nhanh
        c = c + 1 if v else 0
        run[i] = c
    return run


def _win_sum(x: np.ndarray, n: int) -> np.ndarray:
    """Tổng n phần tử kết thúc tại i (gồm i): cs[i] - cs[i-n]. NaN khi i < n."""
    cs = np.concatenate([[0.0], np.cumsum(x)])
    out = np.full(len(x), np.nan)
    out[n - 1:] = cs[n:] - cs[:-n]
    return out


def bar_features(m1: pd.DataFrame) -> pd.DataFrame:
    """Đặc trưng theo nến, chỉ dùng nến <= chính nó. m1 có time, close, gap_min, sắp xếp tăng dần.
    ser*, ret60_z chưa nhân dấu basket (dấu thị trường: > 0 = giá tăng)."""
    t = m1["time"].astype("datetime64[ns]")
    if not t.is_monotonic_increasing:
        raise ValueError("m1 phải sắp xếp tăng dần theo time")
    c = m1["close"].to_numpy(float)
    lc = np.log(c)
    run = _contig_run(m1["gap_min"].to_numpy(float))
    r = np.diff(lc, prepend=np.nan)
    dc = np.diff(c, prepend=np.nan)
    bad = run == 0                        # return qua khoảng trống: không dùng
    r = np.where(bad, 0.0, r)
    dc = np.where(bad, 0.0, dc)

    def valid(n):
        return run >= n

    out = pd.DataFrame({"time": t.to_numpy()})
    rv30 = np.sqrt(_win_sum(r ** 2, RV_WIN))
    out["lrv30"] = np.where(valid(RV_WIN) & (rv30 > 0), np.log(np.where(rv30 > 0, rv30, 1.0)), np.nan)
    for n, name in ((TREND_WIN, "ser60"), (TREND_WIN_SHORT, "ser30")):
        net = _win_sum(dc, n)
        path = _win_sum(np.abs(dc), n)
        out[name] = np.where(valid(n) & (path > 0), net / np.where(path > 0, path, 1.0), np.nan)
    s1, s2 = _win_sum(r, TREND_WIN), _win_sum(r ** 2, TREND_WIN)
    out["ret60_z"] = np.where(valid(TREND_WIN) & (s2 > 0), s1 / np.sqrt(np.where(s2 > 0, s2, 1.0)), np.nan)
    out["hour"] = t.dt.hour.to_numpy()
    return out


def fit_hour_norm(bars: pd.DataFrame, cutoff) -> pd.DataFrame:
    """mu_h, sigma_h của lrv30 theo giờ server, CHỈ trên nến có time < cutoff (đầu tháng test)."""
    b = bars[(bars["time"] < pd.Timestamp(cutoff)) & bars["lrv30"].notna()]
    return b.groupby("hour")["lrv30"].agg(mu="mean", sd="std", n="size")


def apply_hour_norm(lrv: pd.Series, hour: pd.Series, norm: pd.DataFrame) -> pd.Series:
    mu = hour.map(norm["mu"])
    sd = hour.map(norm["sd"])
    return (lrv - mu) / sd


def attach_features(samples: pd.DataFrame, bars: pd.DataFrame, offset=pd.Timedelta(0)) -> pd.DataFrame:
    """Gắn đặc trưng của nến đã đóng gần nhất; nhân dấu basket (BUY +1, SELL -1).
    Trả bản sao samples + m1_bar_time, m1_bar_age_min, hour, lrv30, ser60, ser30, ret60_z, m1_ok."""
    ev = attach_last_closed(samples, bars, time_col="timestamp", cols=(*BAR_COLS, "hour"),
                            offset=offset, prefix="m1_")
    sign = np.where(ev["side"].astype(str).str.upper() == "BUY", 1.0, -1.0)
    out = samples.copy()
    out["m1_bar_time"] = ev["m1_bar_time"]
    out["m1_bar_age_min"] = ev["m1_bar_age_min"]
    out["hour"] = ev["m1_hour"]
    out["lrv30"] = ev["m1_lrv30"]
    for c in ("ser60", "ser30", "ret60_z"):
        out[c] = ev[f"m1_{c}"] * sign
    out.loc[out["m1_bar_age_min"] != 0, ["lrv30", "ser60", "ser30", "ret60_z"]] = np.nan
    out["m1_ok"] = out[["lrv30", "ser60", "ser30", "ret60_z"]].notna().all(axis=1)
    return out


def with_fold_vol(samples: pd.DataFrame, bars: pd.DataFrame, test_month: str):
    """Bản sao samples có vol_z chuẩn hóa bằng mu_h, sigma_h fit trên M1 trước tháng test."""
    norm = fit_hour_norm(bars, pd.Timestamp(test_month + "-01"))
    s = samples.copy()
    s["vol_z"] = apply_hour_norm(s["lrv30"], s["hour"], norm)
    return s, norm


def check_decision_bar(feat: pd.DataFrame) -> int:
    """Số dòng vi phạm: nến dùng phải ĐÓNG trước t, tức bar_time + 1 phút <= t."""
    ok = feat["m1_bar_time"].isna() | (feat["m1_bar_time"] + pd.Timedelta(minutes=1) <= feat["timestamp"])
    return int((~ok).sum())


def check_truncation(m1: pd.DataFrame, cuts) -> float:
    """Đặc trưng nến <= cut tính trên M1 bị cắt tại cut phải trùng bản đầy đủ. Trả max |chênh|."""
    full = bar_features(m1)
    worst = 0.0
    for cut in cuts:
        part = bar_features(m1[m1["time"] <= cut].reset_index(drop=True))
        a = full.loc[full["time"] <= cut, BAR_COLS].to_numpy()
        b = part[list(BAR_COLS)].to_numpy()
        if not np.array_equal(np.isnan(a), np.isnan(b)):
            return np.inf
        d = np.nanmax(np.abs(a - b)) if np.isfinite(a).any() else 0.0
        worst = max(worst, float(d))
    return worst


def check_perturb_future(samples: pd.DataFrame, m1: pd.DataFrame, events_idx, seed=0) -> tuple[int, int]:
    """Với mỗi sự kiện: làm nhiễu MỌI nến có giờ mở >= floor(t) (nến đang chạy + tương lai),
    tính lại đặc trưng. Trả (số sự kiện đặc trưng bị đổi -- phải 0, số sự kiện kế tiếp bị đổi -- nên > 0)."""
    rng = np.random.default_rng(seed)
    cols = ["lrv30", "ser60", "ser30", "ret60_z"]
    base = attach_features(samples, bar_features(m1))
    changed, sensitive = 0, 0
    for i in events_idx:
        t_floor = samples.loc[i, "timestamp"].floor("min")
        m = m1.copy()
        fut = m["time"] >= t_floor
        m.loc[fut, "close"] = m.loc[fut, "close"] * np.exp(rng.normal(0, 2e-3, int(fut.sum())))
        new = attach_features(samples, bar_features(m))
        a, b = base.loc[i, cols].to_numpy(float), new.loc[i, cols].to_numpy(float)
        if not np.allclose(a, b, equal_nan=True, rtol=0, atol=1e-12):
            changed += 1
        # độ nhạy của test: một sự kiện sau t + 2 phút phải đổi
        later = samples.index[(samples["timestamp"] >= t_floor + pd.Timedelta(minutes=3))
                              & base["m1_ok"]]
        if len(later):
            j = later[np.argmin(samples.loc[later, "timestamp"].to_numpy())]
            if not np.allclose(base.loc[j, cols].to_numpy(float), new.loc[j, cols].to_numpy(float),
                               equal_nan=True, rtol=0, atol=1e-12):
                sensitive += 1
    return changed, sensitive


def check_norm_no_test(bars: pd.DataFrame, cutoff, seed=0) -> float:
    """mu_h, sigma_h không đổi khi làm nhiễu lrv30 của nến >= cutoff. Trả max |chênh|."""
    a = fit_hour_norm(bars, cutoff)
    b2 = bars.copy()
    m = b2["time"] >= pd.Timestamp(cutoff)
    b2.loc[m, "lrv30"] = b2.loc[m, "lrv30"] + np.random.default_rng(seed).normal(0, 5, int(m.sum()))
    b = fit_hour_norm(b2, cutoff)
    return float(np.nanmax(np.abs(a[["mu", "sd"]].to_numpy() - b[["mu", "sd"]].to_numpy())))


def steps_last_60m(samples: pd.DataFrame) -> pd.Series:
    """Số bậc của chính basket mở trong (t - 60 phút, t], gồm Rk hiện tại và R1 nếu trong cửa sổ."""
    out = pd.Series(0, index=samples.index, dtype="int64")
    w = np.int64(60 * 60 * 10**9)
    for _, d in samples.groupby("basket_id", sort=False):
        ts = d["timestamp"].to_numpy().astype("datetime64[ns]").astype("int64")
        o = np.argsort(ts)
        tss = ts[o]
        cnt = np.searchsorted(tss, tss, "right") - np.searchsorted(tss, tss - w, "right")
        since_r1 = np.expm1(d["log_since_r1"].to_numpy()[o])     # phút
        cnt = cnt + (since_r1 < 60)
        out.loc[d.index[o]] = cnt
    return out


def walk_forward_m1(samples: pd.DataFrame, bars: pd.DataFrame, test_months, features, **kw):
    """Gọi evaluate.walk_forward từng tháng một; trước mỗi fold tính lại vol_z bằng mu_h, sigma_h
    fit trên M1 < đầu tháng test. Train và test của fold dùng cùng một chuẩn hóa (chỉ từ quá khứ).
    Thứ tự dòng đầu ra giống walk_forward(samples, test_months, ...)."""
    outs = []
    for m in test_months:
        s = with_fold_vol(samples, bars, m)[0] if "vol_z" in (features or []) else samples
        if features is not None and s[features].isna().any().any():
            raise ValueError(f"NaN trong đặc trưng ở fold {m}; lọc dòng trước khi gọi")
        outs.append(walk_forward(s, [m], features=features, **kw))
    return pd.concat(outs, ignore_index=True)


F = types.SimpleNamespace(
    bar_features=bar_features, fit_hour_norm=fit_hour_norm, apply_hour_norm=apply_hour_norm,
    attach_features=attach_features, with_fold_vol=with_fold_vol,
    check_decision_bar=check_decision_bar, check_truncation=check_truncation,
    check_perturb_future=check_perturb_future, check_norm_no_test=check_norm_no_test,
    steps_last_60m=steps_last_60m, walk_forward_m1=walk_forward_m1,
)

Tầng đặc trưng nến 5p M5

In [8]:
BLOCK = pd.Timedelta(minutes=5)
LAM_MAIN = 120          # hazard 1/120 khối ~ 10 giờ (plan G5) -- CHỐT
LAM_SENS = 60           # chỉ độ nhạy
R_MAX = 600
K_CANDIDATES = (5, 10, 20)
K_DEFAULT = 10          # G2; giữ khi các ứng viên hòa trên train
K_TIE = 0.005           # "hòa" = PR-AUC CV kém k tốt nhất < 0,005
HMM_STATES = 3
HMM_NINIT = 5
SEQ_GAP = pd.Timedelta(hours=6)
M5_COLS = ["cp5", "cp10", "cp20", "p_low", "p_mid", "p_high", "drift"]


def m5_blocks(m1: pd.DataFrame) -> pd.DataFrame:
    """Một dòng mỗi khối có >= 1 nến. Cột: time (giờ mở), close_time, n_bars, n_ret,
    valid, reason, lrv5, ret5, hour. Chỉ dùng nến trong khối và nến liền trước (cho return đầu)."""
    t = m1["time"].astype("datetime64[ns]")
    if not t.is_monotonic_increasing:
        raise ValueError("m1 phải sắp xếp tăng dần theo time")
    lc = np.log(m1["close"].to_numpy(float))
    r = np.diff(lc, prepend=np.nan)
    ok = np.nan_to_num(m1["gap_min"].to_numpy(float), nan=0.0) == 1
    r = np.where(ok, r, 0.0)
    d = pd.DataFrame({"T": t.dt.floor("5min").to_numpy(), "r2": r ** 2, "r": r, "ok": ok})
    g = d.groupby("T", sort=True)
    b = pd.DataFrame({"n_bars": g.size(), "n_ret": g["ok"].sum(), "rv": np.sqrt(g["r2"].sum()),
                      "ret5": g["r"].sum()})
    b.index.name = "time"
    b = b.reset_index()
    b["close_time"] = b["time"] + BLOCK
    reason = np.where(b["n_bars"] < 5, "thieu_nen",
                      np.where(b["n_ret"] < 5, "noi_qua_khoang_trong",
                               np.where(b["rv"] <= 0, "rv_bang_0", "ok")))
    b["reason"] = reason
    b["valid"] = reason == "ok"
    b["lrv5"] = np.where(b["valid"], np.log(np.where(b["rv"] > 0, b["rv"], 1.0)), np.nan)
    b.loc[~b["valid"], "ret5"] = np.nan
    b["hour"] = b["time"].dt.hour
    return b.drop(columns="rv")


def decision_block_key(t: pd.Series) -> pd.Series:
    """Giờ mở của khối M5 mới nhất ĐÃ ĐÓNG tại t: floor(t, 5 phút) - 5 phút."""
    return pd.to_datetime(t).astype("datetime64[ns]").dt.floor("5min") - BLOCK


def attach_m5(samples: pd.DataFrame, feat: pd.DataFrame, cols) -> pd.DataFrame:
    """Gắn giá trị khối hợp lệ mới nhất có giờ mở <= key (merge_asof backward, không bao giờ lấy
    khối sau key). block_age_min > 0 (khối cần dùng không hợp lệ) -> NaN. Giữ index/thứ tự samples."""
    f = feat[["time", *cols]].rename(columns={"time": "_blk"}).sort_values("_blk")
    ev = pd.DataFrame({"_key": decision_block_key(samples["timestamp"]).to_numpy(),
                       "_ord": np.arange(len(samples))})
    j = pd.merge_asof(ev.sort_values("_key"), f, left_on="_key", right_on="_blk", direction="backward")
    j = j.sort_values("_ord")
    out = pd.DataFrame(index=samples.index)
    out["m5_block_time"] = j["_blk"].to_numpy()
    out["m5_block_age_min"] = (j["_key"] - j["_blk"]).dt.total_seconds().div(60).to_numpy()
    for c in cols:
        out[c] = np.where(out["m5_block_age_min"] == 0, j[c].to_numpy(), np.nan)
    return out


def fit_hour_norm_m5(blocks: pd.DataFrame, cutoff) -> pd.DataFrame:
    b = blocks[blocks["valid"] & (blocks["time"] < pd.Timestamp(cutoff))]
    return b.groupby("hour")["lrv5"].agg(mu="mean", sd="std", n="size")


def seq_lengths(times: pd.Series) -> list[int]:
    """Độ dài các chuỗi con, tách khi khoảng cách 2 khối hợp lệ liên tiếp > SEQ_GAP."""
    gap = times.diff() > SEQ_GAP
    sid = gap.cumsum().to_numpy()
    return pd.Series(sid).value_counts(sort=False).sort_index().tolist()


def run_bocpd_multi_k(z, prior: NIGPrior, lam: float, ks=K_CANDIDATES, r_max: int = R_MAX) -> np.ndarray:
    """P(r_t <= k | z_1..t) cho nhiều k, dùng lại lớp BOCPD (bocpd.py). Shape (T, len(ks))."""
    model = BOCPD(prior, lam, r_max, k_cp=max(ks))
    out = np.empty((len(z), len(ks)))
    for i, x in enumerate(np.asarray(z, float)):
        model.update(x)
        c = np.cumsum(np.exp(model.log_r))
        out[i] = [c[min(k, c.size - 1)] for k in ks]
    return out


def fold_inputs(blocks: pd.DataFrame, test_month: str) -> dict:
    """Đầu vào của fold: khối hợp lệ từ 02 đến hết tháng test; z chuẩn hóa bằng mu_h, sigma_h fit < cutoff;
    X HMM = (ret5 / sd_ret_train, z). Mọi tham số chỉ từ khối < cutoff."""
    cutoff = pd.Timestamp(test_month + "-01")
    end = cutoff + pd.offsets.MonthBegin(1)
    b = blocks[blocks["valid"] & (blocks["time"] < end)].reset_index(drop=True)
    norm = fit_hour_norm_m5(blocks, cutoff)
    z = ((b["lrv5"] - b["hour"].map(norm["mu"])) / b["hour"].map(norm["sd"])).to_numpy()
    tr = (b["time"] < cutoff).to_numpy()
    sd_ret = float(np.std(b.loc[tr, "ret5"]))
    X = np.column_stack([b["ret5"].to_numpy() / sd_ret, z])
    return dict(b=b, z=z, tr=tr, norm=norm, sd_ret=sd_ret, X=X, cutoff=cutoff,
                lengths=seq_lengths(b["time"]), len_tr=seq_lengths(b.loc[tr, "time"]))


def fold_features(blocks: pd.DataFrame, test_month: str, lam: float = LAM_MAIN, hmm: bool = True,
                  hmm_model=None, seed: int = 0):
    """Đặc trưng M5 theo khối hợp lệ từ 02 đến hết tháng test, chỉ dùng tham số fit < cutoff.
    Trả (feat, info). feat: time, hour, z, ret5, cp5, cp10, cp20 [, p_low, p_mid, p_high, drift].
    hmm_model: truyền mô hình đã fit để bỏ qua bước fit (dùng trong kiểm tra nhiễu tương lai)."""
    I = fold_inputs(blocks, test_month)
    b, z, tr, X = I["b"], I["z"], I["tr"], I["X"]
    prior = NIGPrior.from_train(z[tr])
    cp = run_bocpd_multi_k(z, prior, lam)
    feat = pd.DataFrame({"time": b["time"], "hour": b["hour"], "z": z, "ret5": b["ret5"]})
    for j, k in enumerate(K_CANDIDATES):
        feat[f"cp{k}"] = cp[:, j]
    info = {"norm": I["norm"], "prior": prior, "n_blocks": len(b), "n_train_blocks": int(tr.sum())}
    if hmm:
        model = hmm_model if hmm_model is not None else fit_hmm(X[tr], HMM_STATES, I["len_tr"],
                                                                 n_init=HMM_NINIT, seed=seed)
        probs = forward_filter(model, X, I["lengths"])     # FILTERED P(z_t | x_1..t)
        order, table = map_states(model, vol_col=1)
        pm = probs[:, order]                               # cột 0 = low, 2 = high (theo mean z)
        feat["p_low"], feat["p_mid"], feat["p_high"] = pm[:, 0], pm[:, 1], pm[:, 2]
        feat["drift"] = probs @ model.means_[:, 0]         # Σ p_i μ_i (đơn vị: SD ret5 train), dấu thị trường
        table["occupancy_train"] = pm[tr].mean(axis=0)
        table["occupancy_test"] = pm[~tr].mean(axis=0)
        info.update(model=model, order=order, states=table, sd_ret=I["sd_ret"], X=X, lengths=I["lengths"],
                    train_mask=tr, loglik_train=float(model.score(X[tr], I["len_tr"])) / tr.sum())
    return feat, info


def signed_rows(samples: pd.DataFrame, att: pd.DataFrame, k_main: int) -> pd.DataFrame:
    """Cột đưa vào mô hình: cp = P(r <= k_main), p_high, drift_s = dấu basket . drift (< 0 = ngược basket)."""
    sign = np.where(samples["side"].astype(str).str.upper() == "BUY", 1.0, -1.0)
    out = pd.DataFrame(index=samples.index)
    for k in K_CANDIDATES:
        out[f"cp{k}"] = att[f"cp{k}"]
    out["cp"] = att[f"cp{k_main}"]
    if "p_high" in att:
        out["p_high"] = att["p_high"]
        out["drift_s"] = att["drift"] * sign
    return out


def walk_forward_m5(S: pd.DataFrame, bars: pd.DataFrame, fold_cols: dict, test_months, features, **kw):
    """Mỗi fold m: gắn cột M5 của chính fold m (fold_cols[m], index = S.index) rồi gọi
    features.walk_forward_m1 cho tháng m (vol_z chuẩn hóa theo fold như G4)."""
    outs = []
    for m in test_months:
        end = pd.Timestamp(m + "-01") + pd.offsets.MonthBegin(1)
        s = S[S["timestamp"] < end].copy()     # đặc trưng fold m chỉ tính đến hết tháng m
        extra = [c for c in features if c in fold_cols[m].columns]
        s[extra] = fold_cols[m].loc[s.index, extra]
        outs.append(walk_forward_m1(s, bars, [m], features, **kw))
    return pd.concat(outs, ignore_index=True)


def check_block_closed(samples: pd.DataFrame, att: pd.DataFrame) -> dict:
    """Khối dùng phải đóng trước t: block_time + 5' <= floor(t); báo trễ."""
    used = att["m5_block_age_min"] == 0
    close = att.loc[used, "m5_block_time"] + BLOCK
    t = samples.loc[used, "timestamp"]
    viol = int((close > t.dt.floor("min")).sum())
    lag = (t - close).dt.total_seconds() / 60
    unused = (t.dt.floor("min") - close).dt.total_seconds() / 60       # số nến M1 đã đóng chưa dùng
    return dict(violations=viol, lag_min_max=float(lag.max()), lag_min_mean=float(lag.mean()),
                unused_bars=sorted(unused.astype(int).unique().tolist()))


def check_truncation_m5(m1: pd.DataFrame, cuts) -> tuple[float, int]:
    """Cắt M1 tại cut (giữ nến mở <= cut). Khối đã đóng (T + 4' <= cut) phải trùng bản đầy đủ;
    khối đang chạy (T <= cut < T + 4') phải bị đánh dấu không hợp lệ. Trả (max |chênh|, số khối
    đang chạy bị coi là hợp lệ -- phải 0)."""
    full = m5_blocks(m1).set_index("time")
    worst, leaked = 0.0, 0
    for cut in cuts:
        part = m5_blocks(m1[m1["time"] <= cut].reset_index(drop=True)).set_index("time")
        closed = part.index[part.index + pd.Timedelta(minutes=4) <= cut]
        a = full.loc[closed, ["lrv5", "ret5"]].to_numpy()
        b = part.loc[closed, ["lrv5", "ret5"]].to_numpy()
        if not np.array_equal(np.isnan(a), np.isnan(b)):
            return np.inf, leaked
        if np.isfinite(a).any():
            worst = max(worst, float(np.nanmax(np.abs(a - b))))
        running = part.index[part.index + pd.Timedelta(minutes=4) > cut]
        leaked += int(part.loc[running, "valid"].sum())
    return worst, leaked


M = types.SimpleNamespace(
    m5_blocks=m5_blocks, decision_block_key=decision_block_key, attach_m5=attach_m5,
    fit_hour_norm_m5=fit_hour_norm_m5, seq_lengths=seq_lengths, run_bocpd_multi_k=run_bocpd_multi_k,
    fold_inputs=fold_inputs, fold_features=fold_features, signed_rows=signed_rows,
    walk_forward_m5=walk_forward_m5, check_block_closed=check_block_closed,
    check_truncation_m5=check_truncation_m5,
    BLOCK=BLOCK, LAM_MAIN=LAM_MAIN, LAM_SENS=LAM_SENS, R_MAX=R_MAX, K_CANDIDATES=K_CANDIDATES,
    K_DEFAULT=K_DEFAULT, K_TIE=K_TIE, HMM_STATES=HMM_STATES, HMM_NINIT=HMM_NINIT,
    SEQ_GAP=SEQ_GAP, M5_COLS=M5_COLS,
)

Config Tham số và ngưỡng

In [9]:
SAMPLES_PATH = str(C.INTERIM / "samples.parquet")
M1_PATH = str(C.INTERIM / "m1_clean.parquet")
N_BOOT = 1000
N_PERTURB = 10
# ÉP k của P(r <= k) (dùng để tái tạo bản đóng băng k = 10); None = chọn theo tiêu chí trên train (mục B)
K_MAIN_OVERRIDE = None  # phải thuộc list(M.K_CANDIDATES) nếu đặt khác None

TEST_MONTHS = ["2026-04", "2026-05", "2026-06", "2026-07", "2026-08"]
F1 = list(C.F1_FEATURES)
A3 = F1 + ["vol_z", "ser60"]
REG = ["p_high", "drift_s"]
K_BUCKETS = [(2, 4, "2-4"), (5, 9, "5-9"), (10, 999, ">=10")]
# Ngưỡng nhiễu cảnh báo -- đặt trước (mục D)
NOISE_CP_THR = 0.5          # "báo change point" khi P(r <= 5) > 0,5
NOISE_MAX_ALL = 0.25        # hỏng nếu > 25% khối OOS báo cp
NOISE_MAX_HOUR = 0.50       # hoặc có giờ nào > 50%
HMM_MIN_OCC = 0.02          # hỏng nếu state nào chiếm < 2% train
HMM_MAX_HIGH = 0.60         # hoặc state high chiếm > 60% train
HMM_MIN_AGREE = 0.70        # hoặc nhãn argmax trên tháng 03 trùng fold 08 < 70%

Tools ghép đặc trưng và ghi log theo dõi

In [10]:
class Tee:
    def __init__(self, path):
        self.f = open(path, "w", encoding="utf-8")

    def __call__(self, *a):
        s = " ".join(str(x) for x in a)
        print(s, flush=True)
        self.f.write(s + "\n")
        self.f.flush()


def section(log, title):
    log("\n" + "=" * 70 + f"\n## {title}\n" + "=" * 70)


def fold_cols_for(samples_idx, samples, feats_by_fold, k_main):
    out = {}
    for m, f in feats_by_fold.items():
        cols = [c for c in M.M5_COLS if c in f.columns]
        att = M.attach_m5(samples, f, cols)
        out[m] = M.signed_rows(samples, att, k_main).loc[samples_idx]
    return out

Chuẩn bị thư mục và đọc data

In [11]:
for d in (C.LOGS, C.TABLES, C.FIGS):
    Path(d).mkdir(parents=True, exist_ok=True)
log = Tee(C.LOGS / "g5_report.txt")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
t_start = time.time()

log("G5 REPORT — Họ 4 (change point), họ 5 (regime), ablation bước 2")
section(log, "0. MÔI TRƯỜNG VÀ THAM SỐ CHỐT TRƯỚC")
log(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
    f"sklearn {sklearn.__version__} | hmmlearn {hmmlearn.__version__}")
log(f"n_boot={N_BOOT} | seed={C.SEED} | n_perturb={N_PERTURB}")
log("Khối M5: 5 nến M1 [T, T+4], đóng tại T+5'; hợp lệ khi đủ 5 nến và 5 return 1' liên tục (gồm nến T−1).")
log("Ghép tại t: khối T = floor(t, 5') − 5' → TRỄ: t − giờ đóng khối ∈ [0; 5'), tối đa 4 nến M1 đã đóng chưa dùng.")
log(f"BOCPD: z = log RV5 chuẩn hóa theo giờ (fit < tháng test), prior NIG từ z train, hazard 1/{M.LAM_MAIN} "
    f"(độ nhạy 1/{M.LAM_SENS}), r_max={M.R_MAX}, chạy liên tục từ 02.")
log(f"  Ứng viên k: {M.K_CANDIDATES}; tiêu chí chọn (đặt trước, chỉ train fold 04 = nhãn tháng 03): "
    f"PR-AUC CV 5 nhóm theo ngày của A3 + P(r ≤ k);")
log(f"  chọn k cao nhất, nếu k = {M.K_DEFAULT} (mặc định G2) kém k tốt nhất < {M.K_TIE} thì giữ {M.K_DEFAULT}.")
log(f"HMM: {M.HMM_STATES} state, full cov, {M.HMM_NINIT} khởi tạo (giữ LL train tốt nhất), X = (ret5/sd_train, z), "
    f"tách chuỗi tại gap > {M.SEQ_GAP}; đặc trưng CHỈ từ forward_filter.")
log("  p_high = P(state có mean z cao nhất | x_1..t); drift_s = dấu basket · Σ p_i μ_i(ret) (< 0 = ngược basket).")
log("Ablation bước 2 (cộng dồn trên A3): A4 = A3 + cp; A5 = A4 + p_high + drift_s. Mốc chính: A3.")

samples = pd.read_parquet(SAMPLES_PATH)
m1 = pd.read_parquet(M1_PATH)
m1["time"] = m1["time"].astype("datetime64[ns]")
bars = F.bar_features(m1)

G5 REPORT — Họ 4 (change point), họ 5 (regime), ablation bước 2

## 0. MÔI TRƯỜNG VÀ THAM SỐ CHỐT TRƯỚC
python 3.12.10 | pandas 3.0.6 | numpy 2.5.4 | sklearn 1.9.1 | hmmlearn 0.3.3
n_boot=1000 | seed=42 | n_perturb=10
Khối M5: 5 nến M1 [T, T+4], đóng tại T+5'; hợp lệ khi đủ 5 nến và 5 return 1' liên tục (gồm nến T−1).
Ghép tại t: khối T = floor(t, 5') − 5' → TRỄ: t − giờ đóng khối ∈ [0; 5'), tối đa 4 nến M1 đã đóng chưa dùng.
BOCPD: z = log RV5 chuẩn hóa theo giờ (fit < tháng test), prior NIG từ z train, hazard 1/120 (độ nhạy 1/60), r_max=600, chạy liên tục từ 02.
  Ứng viên k: (5, 10, 20); tiêu chí chọn (đặt trước, chỉ train fold 04 = nhãn tháng 03): PR-AUC CV 5 nhóm theo ngày của A3 + P(r ≤ k);
  chọn k cao nhất, nếu k = 10 (mặc định G2) kém k tốt nhất < 0.005 thì giữ 10.
HMM: 3 state, full cov, 5 khởi tạo (giữ LL train tốt nhất), X = (ret5/sd_train, z), tách chuỗi tại gap > 0 days 06:00:00; đặc trưng CHỈ từ forward_filter.
  p_high = P(state có mean z cao nhất | x_1..t); drift_s = d

Data validation

In [12]:
section(log, "A. KHỐI M5 VÀ DÒNG BỊ LOẠI")
blocks = M.m5_blocks(m1)
q = blocks["reason"].value_counts()
log(f"Khối có ≥ 1 nến: {len(blocks):,} | hợp lệ {int(blocks['valid'].sum()):,} "
    f"({100 * blocks['valid'].mean():.2f}%) | lý do không hợp lệ: {q.drop('ok', errors='ignore').to_dict()}")
inval_h = blocks.loc[~blocks["valid"]].groupby("hour").size()
log(f"  khối không hợp lệ theo giờ server: {inval_h.to_dict()}  (0h/22–23h = mở/đóng phiên)")
qb = blocks.groupby([blocks["time"].dt.strftime("%Y-%m"), "reason"]).size().unstack(fill_value=0)
qb.to_csv(C.TABLES / "g5_blocks_quality.csv")

feat = F.attach_features(samples, bars)                    # G4 (vol, trend)
att_all = M.attach_m5(samples, blocks[blocks["valid"]], ["lrv5"])
m5_ok = att_all["m5_block_age_min"] == 0
g4_ok = feat["m1_ok"]
drop5 = g4_ok & ~m5_ok
log(f"Dòng mẫu {len(samples):,} | loại do G4 (M1): {int((~g4_ok).sum())} | loại THÊM do khối M5 cần dùng "
    f"không hợp lệ: {int(drop5.sum())} ({100 * drop5.mean():.2f}%)")
if drop5.any():
    log(f"  khối bị thiếu theo lý do: "
        f"{blocks.set_index('time').reindex(M.decision_block_key(samples.loc[drop5, 'timestamp']))['reason'].fillna('khong_co_nen').value_counts().to_dict()}")
    log(f"  theo giờ của t: {samples.loc[drop5, 'timestamp'].dt.hour.value_counts().sort_index().to_dict()} | "
        f"dương bị loại: {int(samples.loc[drop5, 'y'].sum())}")
keep = g4_ok & m5_ok
pct_all = 100 * (1 - keep.mean())
log(f"Tổng loại: {int((~keep).sum())} dòng = {pct_all:.2f}% ({'OK < 1%' if pct_all < 1 else '!! > 1%, xem lại'})")
S = feat[keep].copy()
log(f"Tập chung G5 (mọi mô hình đánh giá trên tập này): {len(S):,} dòng, dương {int(S['y'].sum())}")

# Đặc trưng theo fold (chính: hazard 1/120 + HMM)
t0 = time.time()
feats, infos = {}, {}
for m in TEST_MONTHS:
    feats[m], infos[m] = M.fold_features(blocks, m, M.LAM_MAIN, hmm=True, seed=C.SEED)
log(f"\nBOCPD + HMM 5 fold: {time.time() - t0:.0f} s | số khối/fold: "
    f"{ {m: infos[m]['n_blocks'] for m in TEST_MONTHS} }")


## A. KHỐI M5 VÀ DÒNG BỊ LOẠI
Khối có ≥ 1 nến: 41,181 | hợp lệ 40,863 (99.23%) | lý do không hợp lệ: {'thieu_nen': 194, 'noi_qua_khoang_trong': 124}
  khối không hợp lệ theo giờ server: {0: 137, 2: 1, 3: 8, 17: 1, 18: 2, 19: 5, 20: 6, 21: 15, 22: 131, 23: 12}  (0h/22–23h = mở/đóng phiên)
Dòng mẫu 7,889 | loại do G4 (M1): 36 | loại THÊM do khối M5 cần dùng không hợp lệ: 0 (0.00%)
Tổng loại: 36 dòng = 0.46% (OK < 1%)
Tập chung G5 (mọi mô hình đánh giá trên tập này): 7,853 dòng, dương 2115

BOCPD + HMM 5 fold: 320 s | số khối/fold: {'2026-04': 17176, '2026-05': 22901, '2026-06': 28854, '2026-07': 35109, '2026-08': 40863}


Chọn và chốt k cho BOCPD trên train data

In [13]:
section(log, "B. CHỐT k CỦA P(r ≤ k) TRÊN TRAIN (fold 04: nhãn tháng 03, chuẩn hóa/prior fit < 04)")
fc04 = fold_cols_for(S.index, samples, {"2026-04": feats["2026-04"]}, M.K_DEFAULT)["2026-04"]
s04 = F.with_fold_vol(S.join(fc04[[f"cp{k}" for k in M.K_CANDIDATES]]), bars, "2026-04")[0]
tr04, _, _ = make_fold(s04, "2026-04")
tr04 = tr04.reset_index(drop=True)
gkf = GroupKFold(n_splits=5)
rows = []
for name, fs in [("A3", A3)] + [(f"A3+cp{k}", A3 + [f"cp{k}"]) for k in M.K_CANDIDATES]:
    oof = np.zeros(len(tr04))
    for a, b in gkf.split(tr04, tr04["y"], tr04["day"]):
        pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
        pipe.fit(tr04.loc[a, fs], tr04.loc[a, "y"])
        oof[b] = pipe.predict_proba(tr04.loc[b, fs])[:, 1]
    rows.append({"model": name, "cv_pr_auc": pr_auc(tr04["y"], oof)})
ks = pd.DataFrame(rows)
ks.to_csv(C.TABLES / "g5_k_select.csv", index=False)
log(f"Train fold 04: {len(tr04)} dòng, dương {int(tr04['y'].sum())}, {tr04['day'].nunique()} ngày")
log(ks.round(4).to_string(index=False))
cand = ks.iloc[1:].copy()
cand["k"] = list(M.K_CANDIDATES)
best = cand.loc[cand["cv_pr_auc"].idxmax()]
ap10 = float(cand.loc[cand["k"] == M.K_DEFAULT, "cv_pr_auc"].iloc[0])
K_MAIN = M.K_DEFAULT if best["cv_pr_auc"] - ap10 < M.K_TIE else int(best["k"])
if K_MAIN_OVERRIDE is not None:
    log(f"!! k bị ÉP bằng K_MAIN_OVERRIDE {K_MAIN_OVERRIDE} (tiêu chí tự động chọn k = {K_MAIN}); dùng để tái tạo bản đóng băng.")
    K_MAIN = K_MAIN_OVERRIDE
log(f"→ k tốt nhất {int(best['k'])} ({best['cv_pr_auc']:.4f}); k={M.K_DEFAULT}: {ap10:.4f}; "
    f"chênh {best['cv_pr_auc'] - ap10:+.4f} → CHỐT k = {K_MAIN}. Đại diện họ 4: cp = P(r ≤ {K_MAIN}).")
log("  (Chênh CV trên 1 tháng train rất nhỏ so với nhiễu; quy tắc hòa giữ mặc định G2.)")

fold_cols = fold_cols_for(S.index, samples, feats, K_MAIN)
for m in TEST_MONTHS:
    end = pd.Timestamp(m + "-01") + pd.offsets.MonthBegin(1)
    sub = fold_cols[m].loc[S["timestamp"] < end]
    assert not sub.isna().any().any(), f"NaN đặc trưng M5 ở fold {m}"


## B. CHỐT k CỦA P(r ≤ k) TRÊN TRAIN (fold 04: nhãn tháng 03, chuẩn hóa/prior fit < 04)
Train fold 04: 1713 dòng, dương 420, 22 ngày
  model  cv_pr_auc
     A3     0.3232
 A3+cp5     0.3187
A3+cp10     0.3098
A3+cp20     0.3169
→ k tốt nhất 5 (0.3187); k=10: 0.3098; chênh +0.0089 → CHỐT k = 5. Đại diện họ 4: cp = P(r ≤ 5).
  (Chênh CV trên 1 tháng train rất nhỏ so với nhiễu; quy tắc hòa giữ mặc định G2.)


Check tránh future leakage

In [ ]:
section(log, "C. KIỂM TRA CHỐNG NHÌN TƯƠNG LAI (dữ liệu thật)")
rng = np.random.default_rng(C.SEED)
checks = []
att_S = att_all.loc[S.index]
bc = M.check_block_closed(samples.loc[S.index], att_S)
checks.append(("C1 khối dùng đã đóng: block_time + 5' ≤ floor(t) (mọi dòng tập chung)", bc["violations"] == 0,
               f"vi phạm = {bc['violations']}; trễ t − đóng khối: TB {bc['lag_min_mean']:.2f}', "
               f"max {bc['lag_min_max']:.3f}' (< 5'); nến M1 đã đóng chưa dùng ∈ {bc['unused_bars']}"))
v = F.check_decision_bar(feat)
checks.append(("C2 (G4) nến M1 dùng cho vol/trend đã đóng", v == 0, f"vi phạm = {v}"))
cuts = pd.to_datetime(rng.choice(m1["time"].to_numpy()[20000:], 10, replace=False))
d, leaked = M.check_truncation_m5(m1, cuts)
checks.append(("C3 cắt M1 tại 10 mốc: khối đã đóng không đổi; khối đang chạy bị đánh dấu không hợp lệ",
               d == 0 and leaked == 0, f"max |chênh| = {d:g}; khối đang chạy lọt = {leaked}"))

# C4 nhiễu M1 từ floor(t) trở đi
t0 = time.time()
S_oos = S[S["month"].isin(TEST_MONTHS)]
per_m = max(1, N_PERTURB // len(TEST_MONTHS))
ev_idx = []
for m in TEST_MONTHS:
    pool = S_oos.index[S_oos["month"] == m]
    ev_idx += list(rng.choice(pool, per_m, replace=False))
cols5 = ["cp5", "cp10", "cp20", "p_high", "drift"]
changed, sens, n_sens = 0, 0, 0
for i in ev_idx:
    m = samples.loc[i, "month"]
    t_floor = samples.loc[i, "timestamp"].floor("min")
    mp = m1.copy()
    fut = mp["time"] >= t_floor
    mp.loc[fut, "close"] = mp.loc[fut, "close"] * np.exp(rng.normal(0, 2e-3, int(fut.sum())))
    fp, _ = M.fold_features(M.m5_blocks(mp), m, M.LAM_MAIN, hmm=True, hmm_model=infos[m]["model"])
    base = M.attach_m5(samples.loc[[i]], feats[m], cols5).iloc[0][cols5].to_numpy(float)
    new = M.attach_m5(samples.loc[[i]], fp, cols5).iloc[0][cols5].to_numpy(float)
    if not np.allclose(base, new, rtol=0, atol=1e-12, equal_nan=True):
        changed += 1
    later = S.index[(S["timestamp"] >= t_floor + pd.Timedelta(minutes=10)) & (S["month"] == m)]
    if len(later):
        n_sens += 1
        j = later[np.argmin(S.loc[later, "timestamp"].to_numpy())]
        b2 = M.attach_m5(samples.loc[[j]], feats[m], cols5).iloc[0][cols5].to_numpy(float)
        n2 = M.attach_m5(samples.loc[[j]], fp, cols5).iloc[0][cols5].to_numpy(float)
        sens += int(not np.allclose(b2, n2, rtol=0, atol=1e-12, equal_nan=True))
checks.append((f"C4 nhiễu mọi nến ≥ floor(t), {len(ev_idx)} sự kiện OOS (mỗi tháng {per_m}): "
               "cp5/cp10/cp20/p_high/drift tại t không đổi", changed == 0, f"bị đổi = {changed}"))
checks.append(("C5 độ nhạy của C4: sự kiện kế tiếp (≥ floor(t) + 10') có đổi", sens > 0 and sens == n_sens,
               f"{sens}/{n_sens}  ({time.time() - t0:.0f} s)"))

worst_norm, worst_x, worst_par = 0.0, 0.0, 0.0
for m in TEST_MONTHS:
    cutoff = pd.Timestamp(m + "-01")
    bp = blocks.copy()
    fm = bp["time"] >= cutoff
    bp.loc[fm, "lrv5"] += rng.normal(0, 3, int(fm.sum()))
    bp.loc[fm, "ret5"] += rng.normal(0, 1e-2, int(fm.sum()))
    a, b = M.fit_hour_norm_m5(blocks, cutoff), M.fit_hour_norm_m5(bp, cutoff)
    worst_norm = max(worst_norm, float(np.nanmax(np.abs(a[["mu", "sd"]].to_numpy() - b[["mu", "sd"]].to_numpy()))))
    Ia, Ib = M.fold_inputs(blocks, m), M.fold_inputs(bp, m)
    worst_x = max(worst_x, float(np.max(np.abs(Ia["X"][Ia["tr"]] - Ib["X"][Ib["tr"]]))))
    if m in (TEST_MONTHS[0], TEST_MONTHS[-1]):      # refit HMM thật ở 2 fold (tốn thời gian)
        mb = fit_hmm(Ib["X"][Ib["tr"]], M.HMM_STATES, Ib["len_tr"], n_init=M.HMM_NINIT, seed=C.SEED)
        ma = infos[m]["model"]
        worst_par = max(worst_par, float(np.max(np.abs(ma.means_ - mb.means_))),
                        float(np.max(np.abs(ma.transmat_ - mb.transmat_))))
checks.append(("C6 nhiễu khối ≥ tháng test: μ_h,σ_h, X_train HMM (5 fold) và tham số HMM refit (fold 04, 08) "
               "không đổi (HMM refit: dung sai 1e-8 vì làm tròn số học khác máy)", worst_norm == 0 and worst_x == 0 and worst_par < 1e-8,
               f"max |chênh| norm = {worst_norm:g}, X = {worst_x:g}, HMM = {worst_par:g}"))

# C7 filtered vs smoothed (smoothed CHỈ để chứng minh khác biệt)
i8 = infos[TEST_MONTHS[-1]]
X8, L8, mod8 = i8["X"], i8["lengths"], i8["model"]
filt, ll = forward_filter(mod8, X8, L8, return_loglik=True)
smooth = mod8.predict_proba(X8, L8)                  # smoothed -- KHÔNG dùng làm đặc trưng
ends = np.cumsum(L8) - 1
interior = np.setdiff1d(np.arange(len(X8)), ends)
d_end = float(np.max(np.abs(filt[ends] - smooth[ends])))
d_int = float(np.mean(np.abs(filt[interior] - smooth[interior]).max(axis=1)))
d_ll = abs(ll - mod8.score(X8, L8))
same = float(np.max(np.abs(feats[TEST_MONTHS[-1]]["p_high"].to_numpy() - filt[:, i8["order"][-1]])))
checks.append(("C7 đặc trưng = filtered; filtered = smoothed chỉ tại điểm cuối chuỗi, khác ở giữa",
               same == 0 and d_end < 1e-8 and d_int > 1e-3 and d_ll < 1e-6,
               f"|p_high − filtered| = {same:g}; cuối chuỗi {d_end:.1e}; giữa chuỗi TB max|Δ| {d_int:.3f}; "
               f"|LL forward − hmmlearn| = {d_ll:.1e}"))
for name, ok, note in checks:
    log(f"  [{'OK' if ok else '!!'}] {name}  {note}")
pd.DataFrame(checks, columns=["check", "ok", "note"]).to_csv(C.TABLES / "g5_leak_checks.csv", index=False)


## C. KIỂM TRA CHỐNG NHÌN TƯƠNG LAI (dữ liệu thật)


Kiểm định độ ổn định của BOCPD và HMM

In [ ]:
section(log, "D. NHIỄU CẢNH BÁO (ngưỡng đặt trước) VÀ QUYẾT ĐỊNH C3")
oos_blk = pd.concat([feats[m][feats[m]["time"].dt.strftime("%Y-%m") == m].assign(month=m) for m in TEST_MONTHS])
oos_blk["alarm"] = oos_blk["cp5"] > NOISE_CP_THR
nh = oos_blk.groupby("hour").agg(n=("cp5", "size"), alarm_rate=("alarm", "mean"), mean_cp5=("cp5", "mean"),
                                 mean_cp=(f"cp{K_MAIN}", "mean"), mean_z=("z", "mean"))
nh.to_csv(C.TABLES / "g5_bocpd_noise_by_hour.csv")
rate_all = float(oos_blk["alarm"].mean())
rate_hmax = float(nh["alarm_rate"].max())
log(f"[D1] BOCPD (OOS, mỗi tháng từ fold của nó): tỷ lệ khối P(r ≤ 5) > {NOISE_CP_THR}: {100 * rate_all:.2f}% "
    f"| giờ cao nhất {int(nh['alarm_rate'].idxmax())}h = {100 * rate_hmax:.2f}% | ngưỡng hỏng: > "
    f"{100 * NOISE_MAX_ALL:.0f}% tổng hoặc > {100 * NOISE_MAX_HOUR:.0f}% một giờ")
log(f"     mean z OOS theo giờ: min {nh['mean_z'].min():+.2f}, max {nh['mean_z'].max():+.2f} "
    f"(≈ 0 nếu mùa vụ đã khử)")
log("     theo giờ:\n" + nh.round(3).to_string())
alarm_runs = (oos_blk["alarm"] & ~oos_blk["alarm"].shift(fill_value=False)).sum()
days_oos = oos_blk["time"].dt.normalize().nunique()
log(f"     số đợt báo (chuỗi khối liên tiếp > ngưỡng): {int(alarm_runs)} trên {days_oos} ngày "
    f"= {alarm_runs / days_oos:.2f} đợt/ngày (hazard 1/120 khối ≈ 2,3 cp/ngày theo prior)")
bocpd_bad = rate_all > NOISE_MAX_ALL or rate_hmax > NOISE_MAX_HOUR

st_rows, agree_rows = [], []
lab_ref = None
mar = None
for m in reversed(TEST_MONTHS):
    f = feats[m]
    tb = infos[m]["states"].copy()
    tb.insert(0, "fold", m)
    tb["loglik_train_per_block"] = infos[m]["loglik_train"]
    st_rows.append(tb)
    fm = f[f["time"].dt.strftime("%Y-%m") == "2026-03"]
    lab = np.argmax(fm[["p_low", "p_mid", "p_high"]].to_numpy(), axis=1)
    if lab_ref is None:
        lab_ref, mar = lab, fm
    ftest = f[f["time"].dt.strftime("%Y-%m") == m]
    agree_rows.append({"fold": m, "agree_argmax_03_vs_fold08": float((lab == lab_ref).mean()),
                       "corr_p_high_03_vs_fold08": float(np.corrcoef(fm["p_high"], mar["p_high"])[0, 1]),
                       "mean_p_high_03": float(fm["p_high"].mean()),
                       "mean_p_high_test": float(ftest["p_high"].mean()),
                       "mean_drift_test": float(ftest["drift"].mean()),
                       "corr_drift_p_high_test": float(np.corrcoef(ftest["drift"], ftest["p_high"])[0, 1])})
states = pd.concat(st_rows[::-1], ignore_index=True)
stab = pd.DataFrame(agree_rows[::-1])
states.to_csv(C.TABLES / "g5_hmm_states.csv", index=False)
stab.to_csv(C.TABLES / "g5_hmm_stability.csv", index=False)
log("\n[D2] HMM — mapping state theo fold (mean_0/var_0: ret5/sd_train; mean_1/var_1: z; "
    "expected_duration theo khối M5; orig_state = chỉ số state của hmmlearn)")
log(states.round(3).to_string(index=False))
log("\n[D3] HMM — độ ổn định giữa fold (trên tháng 03, chung cho mọi fold; p_high trung bình theo fold)")
log(stab.round(3).to_string(index=False))
occ_min = float(states[["occupancy_train"]].min().iloc[0])
high_max = float(states.loc[states["label"] == "high", "occupancy_train"].max())
agree_min = float(stab["agree_argmax_03_vs_fold08"].min())
hmm_bad = occ_min < HMM_MIN_OCC or high_max > HMM_MAX_HIGH or agree_min < HMM_MIN_AGREE
log(f"     occupancy train nhỏ nhất {occ_min:.3f} (ngưỡng ≥ {HMM_MIN_OCC}); high lớn nhất {high_max:.3f} "
    f"(≤ {HMM_MAX_HIGH}); trùng nhãn tháng 03 nhỏ nhất {agree_min:.3f} (≥ {HMM_MIN_AGREE})")
log(f"\n[D4] Quyết định C3: BOCPD {'HỎNG → CUSUM' if bocpd_bad else 'đạt (giữ BOCPD)'}; "
    f"HMM {'HỎNG → bỏ họ 5' if hmm_bad else 'đạt (giữ HMM)'}.")
if bocpd_bad or hmm_bad:
    log("     !! Phương án dự phòng C3 cần được áp dụng — xem plan.md mục 4.")


## D. NHIỄU CẢNH BÁO (ngưỡng đặt trước) VÀ QUYẾT ĐỊNH C3
[D1] BOCPD (OOS, mỗi tháng từ fold của nó): tỷ lệ khối P(r ≤ 5) > 0.5: 1.32% | giờ cao nhất 3h = 3.70% | ngưỡng hỏng: > 25% tổng hoặc > 50% một giờ
     mean z OOS theo giờ: min -0.67, max -0.08 (≈ 0 nếu mùa vụ đã khử)
     theo giờ:
         n  alarm_rate  mean_cp5  mean_cp  mean_z
hour                                             
0     1177       0.018     0.063    0.063  -0.670
1     1284       0.009     0.053    0.053  -0.646
2     1291       0.009     0.058    0.058  -0.428
3     1296       0.037     0.091    0.091  -0.083
4     1296       0.005     0.050    0.050  -0.103
5     1296       0.008     0.054    0.054  -0.134
6     1296       0.006     0.053    0.053  -0.465
7     1296       0.012     0.059    0.059  -0.290
8     1296       0.010     0.053    0.053  -0.289
9     1296       0.004     0.052    0.052  -0.423
10    1296       0.019     0.063    0.063  -0.314
11    1296       0.010     0.057    0.057  -0.455
12    12

So sánh prauc của các nhóm đặc trưng

In [ ]:
section(log, "E. ABLATION ĐẦY ĐỦ — PR-AUC GỘP OOS 04–08 (cùng tập dòng)")
MODELS = {
    "A0 log(k)": ["logk"],
    "A1 họ 1": F1,
    "A2 họ 1 + vol": F1 + ["vol_z"],
    "A3 họ 1 + vol + trend": A3,
    "A4 A3 + change point": A3 + ["cp"],
    "A5 A4 + regime": A3 + ["cp"] + REG,
    "B5 A3 + regime (không cp)": A3 + REG,
}
COMPARE = [("A1 họ 1", "A0 log(k)"), ("A2 họ 1 + vol", "A1 họ 1"), ("A3 họ 1 + vol + trend", "A2 họ 1 + vol"),
           ("A4 A3 + change point", "A3 họ 1 + vol + trend"), ("A5 A4 + regime", "A4 A3 + change point"),
           ("A5 A4 + regime", "A3 họ 1 + vol + trend"), ("B5 A3 + regime (không cp)", "A3 họ 1 + vol + trend"),
           ("A3 họ 1 + vol + trend", "A0 log(k)"), ("A5 A4 + regime", "A0 log(k)")]
oos = {name: M.walk_forward_m5(S, bars, fold_cols, TEST_MONTHS, fs) for name, fs in MODELS.items()}
ref = oos["A0 log(k)"]
for name, o in oos.items():
    assert (o["basket_id"].to_numpy() == ref["basket_id"].to_numpy()).all() and \
           (o["k"].to_numpy() == ref["k"].to_numpy()).all(), f"lệch thứ tự dòng: {name}"
full_g1 = walk_forward(samples, TEST_MONTHS, features=["logk"])
g1_ap = pr_auc(full_g1["y"], full_g1["score"])
log(f"OOS tập chung: {len(ref):,} dòng | dương {int(ref['y'].sum())} | prevalence {ref['y'].mean():.4f} | "
    f"{ref['day'].nunique()} ngày (cụm bootstrap)")
log(f"Đối chiếu G1 (tập đầy đủ {len(full_g1):,} dòng): PR-AUC log(k) = {g1_ap:.4f} (G1 report: 0.4286)")

a3s = oos["A3 họ 1 + vol + trend"]["score"]
rows = []
for name, o in oos.items():
    bs = paired_bootstrap(o["y"], o["score"], a3s, o["day"], n_boot=N_BOOT)
    rows.append({"model": name, "features": " + ".join(MODELS[name]), "n_feat": len(MODELS[name]),
                 "ap": bs["ap_a"], "ci_lo": bs["ci_a"][0], "ci_hi": bs["ci_a"][1]})
tab = pd.DataFrame(rows)
rows = []
for a, b in COMPARE:
    bs = paired_bootstrap(oos[a]["y"], oos[a]["score"], oos[b]["score"], oos[a]["day"], n_boot=N_BOOT)
    rows.append({"so_sanh": f"{a}  −  {b}", "delta": bs["delta"], "ci_lo": bs["ci_delta"][0],
                 "ci_hi": bs["ci_delta"][1], "P(delta>0)": bs["frac_delta_pos"],
                 "CI_chua_0": bs["ci_delta"][0] <= 0 <= bs["ci_delta"][1]})
dtab = pd.DataFrame(rows)
tab.to_csv(C.TABLES / "g5_ablation_ap.csv", index=False)
dtab.to_csv(C.TABLES / "g5_ablation_delta.csv", index=False)
log("\n[E1] PR-AUC gộp (CI 95% bootstrap theo ngày)")
log(tab.drop(columns="features").round(4).to_string(index=False))
log("\n[E2] ΔPR-AUC ghép cặp (CI 95% bootstrap theo ngày, cùng mẫu lặp)")
log(dtab.round(4).to_string(index=False))

bym = pd.DataFrame({name: o.groupby("month").apply(lambda d: pr_auc(d["y"], d["score"]), include_groups=False)
                    for name, o in oos.items()})
bym.insert(0, "prevalence", ref.groupby("month")["y"].mean())
bym.to_csv(C.TABLES / "g5_ablation_by_month.csv")
log("\n[E3] PR-AUC theo tháng (mỗi tháng ~50–60 basket dương — nhiễu)")
log(bym.round(4).to_string())
for a in ("A4 A3 + change point", "A5 A4 + regime", "B5 A3 + regime (không cp)"):
    log(f"  số tháng {a.split()[0]} > A3: {int((bym[a] > bym['A3 họ 1 + vol + trend']).sum())}/5")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
yy = np.arange(len(tab))
ax.errorbar(tab["ap"], yy, xerr=[tab["ap"] - tab["ci_lo"], tab["ci_hi"] - tab["ap"]], fmt="o", capsize=3)
ax.axvline(ref["y"].mean(), color="grey", ls=":", label="prevalence")
ax.axvline(tab.loc[tab["model"].str.startswith("A3"), "ap"].iloc[0], color="C1", ls="--", lw=0.8, label="A3")
ax.set_yticks(yy, tab["model"])
ax.invert_yaxis()
ax.set_xlabel("PR-AUC gộp OOS 04–08 (CI 95% theo ngày)")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(C.FIGS / "g5_ablation.png", dpi=130)
plt.close(fig)


## E. ABLATION ĐẦY ĐỦ — PR-AUC GỘP OOS 04–08 (cùng tập dòng)
OOS tập chung: 6,140 dòng | dương 1695 | prevalence 0.2761 | 100 ngày (cụm bootstrap)
Đối chiếu G1 (tập đầy đủ 6,160 dòng): PR-AUC log(k) = 0.4286 (G1 report: 0.4286)

[E1] PR-AUC gộp (CI 95% bootstrap theo ngày)
                    model  n_feat     ap  ci_lo  ci_hi
                A0 log(k)       1 0.4266 0.3585 0.4850
                  A1 họ 1       4 0.4212 0.3546 0.4796
            A2 họ 1 + vol       5 0.4208 0.3537 0.4792
    A3 họ 1 + vol + trend       6 0.4496 0.3791 0.5113
     A4 A3 + change point       7 0.4466 0.3744 0.5084
           A5 A4 + regime       9 0.4371 0.3646 0.4997
B5 A3 + regime (không cp)       8 0.4396 0.3676 0.5012

[E2] ΔPR-AUC ghép cặp (CI 95% bootstrap theo ngày, cùng mẫu lặp)
                                            so_sanh   delta   ci_lo   ci_hi  P(delta>0)  CI_chua_0
                              A1 họ 1  −  A0 log(k) -0.0054 -0.0142  0.0035       0.131       True
                     

Phân tầng theo hiệu quả của a4 a5 theo độ sâu k, so sánh với a3

In [ ]:
section(log, "F. PHÂN TẦNG THEO ĐỘ SÂU (so với A3)")
rows = []
a3 = oos["A3 họ 1 + vol + trend"]
for name in ("A4 A3 + change point", "A5 A4 + regime"):
    o = oos[name]
    for lo, hi, lab in K_BUCKETS:
        msk = ((ref["k"] >= lo) & (ref["k"] <= hi)).to_numpy()
        bs = paired_bootstrap(o["y"][msk], o["score"][msk], a3["score"][msk], ref["day"][msk], n_boot=N_BOOT)
        rows.append({"model": name.split()[0], "k": lab, "n": int(msk.sum()), "pos": int(ref["y"][msk].sum()),
                     "ap_A3": bs["ap_b"], "ap_model": bs["ap_a"], "delta": bs["delta"],
                     "ci_lo": bs["ci_delta"][0], "ci_hi": bs["ci_delta"][1]})
st = pd.DataFrame(rows)
st.to_csv(C.TABLES / "g5_by_k.csv", index=False)
log(st.round(4).to_string(index=False))


## F. PHÂN TẦNG THEO ĐỘ SÂU (so với A3)
model    k    n  pos  ap_A3  ap_model   delta   ci_lo   ci_hi
   A4  2-4 3606  684 0.2321    0.2312 -0.0009 -0.0054  0.0041
   A4  5-9 1522  519 0.4052    0.4007 -0.0046 -0.0156  0.0086
   A4 >=10 1012  492 0.6047    0.6015 -0.0033 -0.0152  0.0053
   A5  2-4 3606  684 0.2321    0.2274 -0.0047 -0.0120  0.0038
   A5  5-9 1522  519 0.4052    0.3929 -0.0124 -0.0272  0.0028
   A5 >=10 1012  492 0.6047    0.5815 -0.0233 -0.0449 -0.0065


Mô phỏng brake

In [ ]:
section(log, "G. MÔ PHỎNG PHANH (ngưỡng chọn trên train) — CẬN TRÊN, KHÔNG PHẢI GIẢM LỖ")
rows = []
for name in ("A0 log(k)", "A3 họ 1 + vol + trend", "A4 A3 + change point", "A5 A4 + regime"):
    o = oos[name]
    for q in C.BRAKE_RATES:
        rows.append({"model": name.split()[0], "target": q, **brake_sim(o, o[f"thr_{round(q * 100)}"])})
bt = pd.DataFrame(rows)
bt.to_csv(C.TABLES / "g5_brake.csv", index=False)
show = bt[["model", "target", "brake_rate_rows", "halted_pct", "deep_caught", "deep_caught_pct",
           "avoided_total", "avoided_pct", "false_halts", "false_halts_per100"]].copy()
for c in ("brake_rate_rows", "halted_pct", "deep_caught_pct", "avoided_pct"):
    show[c] = 100 * show[c]
log(show.round(2).to_string(index=False))
log("Tỷ lệ phanh thực tế (brake_rate_rows) khác mục tiêu và khác giữa mô hình; so sánh công bằng ở G6.")


## G. MÔ PHỎNG PHANH (ngưỡng chọn trên train) — CẬN TRÊN, KHÔNG PHẢI GIẢM LỖ
model  target  brake_rate_rows  halted_pct  deep_caught  deep_caught_pct  avoided_total  avoided_pct  false_halts  false_halts_per100
   A0     0.1            12.48        5.43           42            15.00            769        12.48           64                3.28
   A0     0.2            24.09       12.25           81            28.93           1485        24.11          158                8.10
   A0     0.3            33.63       19.07          125            44.64           2071        33.62          247               12.66
   A3     0.1            10.54        5.59           51            18.21            869        14.11           58                2.97
   A3     0.2            20.55       11.38           88            31.43           1485        24.11          134                6.87
   A3     0.3            32.02       18.86          122            43.57           2116        34.35          246     

tính độ ổn định của hệ số logistic, và mức độ tương quan spearman giữa các đặc trưng

In [ ]:
section(log, "H. HỆ SỐ LOGISTIC CHUẨN HÓA (A5, từng fold; log-odds / 1 SD train)")
feats5 = MODELS["A5 A4 + regime"]
coefs = {}
for m in TEST_MONTHS:
    s = F.with_fold_vol(S.join(fold_cols[m][["cp", *REG]]), bars, m)[0]
    tr, _, _ = make_fold(s, m)
    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)).fit(tr[feats5], tr["y"])
    coefs[m] = pd.Series(pipe[-1].coef_[0], index=feats5)
ct = pd.DataFrame(coefs)
ct["mean"] = ct[TEST_MONTHS].mean(axis=1)
ct["sign_stable"] = np.sign(ct[TEST_MONTHS]).nunique(axis=1) == 1
ct.to_csv(C.TABLES / "g5_coef.csv")
log(ct.round(3).to_string())
corr = S.join(fold_cols[TEST_MONTHS[-1]][["cp", *REG]])
corr = F.with_fold_vol(corr, bars, TEST_MONTHS[-1])[0]
log("\nSpearman giữa đặc trưng (fold 08, mọi dòng ≤ 08; mô tả dư thừa):")
log(corr[["logk", "vol_z", "ser60", "cp", "p_high", "drift_s"]].corr("spearman").round(2).to_string())


## H. HỆ SỐ LOGISTIC CHUẨN HÓA (A5, từng fold; log-odds / 1 SD train)
              2026-04  2026-05  2026-06  2026-07  2026-08   mean  sign_stable
logk            0.310    0.085    0.336    0.565    0.514  0.362         True
log_gap        -0.062   -0.092   -0.044    0.028    0.027 -0.029        False
accel           0.010    0.029    0.012   -0.026   -0.025  0.000        False
log_since_r1    0.021    0.385    0.142   -0.133   -0.094  0.064        False
vol_z          -0.034   -0.084   -0.049   -0.049   -0.008 -0.045         True
ser60          -0.186   -0.236   -0.219   -0.256   -0.247 -0.229         True
cp             -0.085    0.003    0.036    0.054    0.055  0.013        False
p_high          0.072    0.083    0.070    0.050    0.015  0.058         True
drift_s        -0.205   -0.117   -0.038   -0.025   -0.041 -0.085         True

Spearman giữa đặc trưng (fold 08, mọi dòng ≤ 08; mô tả dư thừa):
         logk  vol_z  ser60    cp  p_high  drift_s
logk     1.00   0.06  -0.51  0.0

Kiểm tra xem kết quả có ổn định khi thay đổi tham số bocpd hoặc bổ sung các đặc trưng regime

In [ ]:
section(log, "I. ĐỘ NHẠY (CHỈ BÁO CÁO — KHÔNG DÙNG CHỌN MÔ HÌNH)")
t0 = time.time()
feats60 = {m: M.fold_features(blocks, m, M.LAM_SENS, hmm=False)[0] for m in TEST_MONTHS}
fc60 = fold_cols_for(S.index, samples, feats60, K_MAIN)
log(f"BOCPD hazard 1/{M.LAM_SENS}: {time.time() - t0:.0f} s")
variants = {f"A3 + cp{k} (1/{M.LAM_MAIN})": (fold_cols, A3 + [f"cp{k}"]) for k in M.K_CANDIDATES}
variants[f"A3 + cp{K_MAIN} (1/{M.LAM_SENS})"] = (fc60, A3 + ["cp"])
variants["A3 + p_high"] = (fold_cols, A3 + ["p_high"])
variants["A3 + drift_s"] = (fold_cols, A3 + ["drift_s"])
rows = []
for name, (fc, fs) in variants.items():
    o = M.walk_forward_m5(S, bars, fc, TEST_MONTHS, fs)
    bs = paired_bootstrap(o["y"], o["score"], a3["score"], o["day"], n_boot=N_BOOT)
    rows.append({"variant": name, "ap": bs["ap_a"], "delta_vs_A3": bs["delta"],
                 "ci_lo": bs["ci_delta"][0], "ci_hi": bs["ci_delta"][1]})
sens_t = pd.DataFrame(rows)
oos60 = pd.concat([feats60[m][feats60[m]["time"].dt.strftime("%Y-%m") == m] for m in TEST_MONTHS])
log(sens_t.round(4).to_string(index=False))
log(f"Tỷ lệ khối OOS P(r ≤ 5) > {NOISE_CP_THR} với hazard 1/{M.LAM_SENS}: {100 * (oos60['cp5'] > NOISE_CP_THR).mean():.2f}%")
sens_t.to_csv(C.TABLES / "g5_sensitivity.csv", index=False)


## I. ĐỘ NHẠY (CHỈ BÁO CÁO — KHÔNG DÙNG CHỌN MÔ HÌNH)
BOCPD hazard 1/60: 115 s
          variant     ap  delta_vs_A3   ci_lo   ci_hi
 A3 + cp5 (1/120) 0.4466      -0.0030 -0.0081  0.0016
A3 + cp10 (1/120) 0.4484      -0.0013 -0.0031  0.0004
A3 + cp20 (1/120) 0.4500       0.0004 -0.0074  0.0100
  A3 + cp5 (1/60) 0.4474      -0.0022 -0.0077  0.0026
      A3 + p_high 0.4486      -0.0010 -0.0040  0.0024
     A3 + drift_s 0.4401      -0.0096 -0.0153 -0.0042
Tỷ lệ khối OOS P(r ≤ 5) > 0.5 với hazard 1/60: 2.05%


In plot

In [ ]:
section(log, "J. HÌNH")
f8 = feats[TEST_MONTHS[-1]].copy()
w0, w1 = pd.Timestamp("2026-08-04 12:00"), pd.Timestamp("2026-08-06 12:00")
fw = f8[(f8["time"] >= w0) & (f8["time"] < w1)]
pw = m1[(m1["time"] >= w0) & (m1["time"] < w1)]
sell = samples[samples["basket_id"] == "BMSFJA9ZX"]
buys = samples[(samples["day"] == pd.Timestamp("2026-08-05")) & (samples["side"] == "BUY")]
fig, axs = plt.subplots(3, 1, figsize=(10, 7.5), sharex=True, gridspec_kw={"height_ratios": [1.2, 1, 1]})
axs[0].plot(pw["time"], pw["close"], lw=0.7, color="k")
for t in sell["timestamp"]:
    axs[0].axvline(t, color="C3", lw=0.4, alpha=0.5)
axs[0].plot([], [], color="C3", lw=1, label=f"SELL BMSFJA9ZX R2–R{int(sell['k'].max())}")
axs[0].scatter(buys["timestamp"], np.interp(buys["timestamp"].astype("int64"), pw["time"].astype("int64"),
                                             pw["close"]), s=8, color="C0", label="bậc ≥ R2 basket BUY 05/08")
axs[0].set_ylabel("XAUUSD M1 close")
axs[0].legend(fontsize=7, loc="upper left")
axs[1].plot(fw["time"], fw["cp5"], lw=0.8, label="P(r ≤ 5)")
axs[1].plot(fw["time"], fw[f"cp{K_MAIN}"], lw=0.8, label=f"P(r ≤ {K_MAIN}) (đại diện)")
axs[1].plot(fw["time"], fw["cp20"], lw=0.6, alpha=0.6, label="P(r ≤ 20)")
axs[1].axhline(NOISE_CP_THR, color="grey", ls=":", lw=0.7)
axs[1].set_ylabel("BOCPD (filtered, 1/120)")
axs[1].legend(fontsize=7, loc="upper left")
axs[2].stackplot(fw["time"], fw["p_low"], fw["p_mid"], fw["p_high"], labels=["low", "mid", "high"],
                 colors=["#9ecae1", "#fdd49e", "#ef6548"], alpha=0.9)
axs[2].set_ylabel("HMM filtered P(state)")
axs[2].legend(fontsize=7, loc="upper left", ncol=3)
axs[2].set_xlabel("giờ server (fold 08: chuẩn hóa/HMM fit trên 02–07)")
fig.suptitle("Case 05/08/2026: change point và regime (đặc trưng tại khối M5 đã đóng)")
fig.tight_layout()
fig.savefig(C.FIGS / "g5_case_0508.png", dpi=130)
plt.close(fig)
dd = fw[(fw["time"] >= "2026-08-05 03:00") & (fw["time"] < "2026-08-05 18:00")]
log(f"05/08 03–18h (basket SELL sâu): mean P(r ≤ 5) = {dd['cp5'].mean():.3f}, max {dd['cp5'].max():.3f}; "
    f"mean p_high = {dd['p_high'].mean():.3f}; tỷ lệ khối argmax = high: "
    f"{(dd[['p_low', 'p_mid', 'p_high']].to_numpy().argmax(1) == 2).mean():.3f}; mean drift = {dd['drift'].mean():+.4f}")
rr = fold_cols[TEST_MONTHS[-1]].loc[S.index.intersection(sell.index)]
log(f"Basket SELL BMSFJA9ZX trong tập chung: {len(rr)} dòng; cp TB {rr['cp'].mean():.3f}; "
    f"p_high TB {rr['p_high'].mean():.3f}; drift_s TB {rr['drift_s'].mean():+.4f}")

fig, ax = plt.subplots(figsize=(7, 3.3))
ax.bar(nh.index, 100 * nh["alarm_rate"], color="C0", label=f"% khối P(r ≤ 5) > {NOISE_CP_THR}")
ax.plot(nh.index, 100 * nh["mean_cp5"], "o-", color="C3", ms=3, label="mean P(r ≤ 5) (%)")
ax.axhline(100 * NOISE_MAX_HOUR, color="grey", ls=":", lw=0.8, label="ngưỡng hỏng theo giờ")
ax.set_xlabel("giờ server")
ax.set_ylabel("%")
ax.set_title("BOCPD OOS 04–08: báo change point theo giờ (mùa vụ còn sót?)")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(C.FIGS / "g5_bocpd_noise_by_hour.png", dpi=130)
plt.close(fig)
for p in ("g5_case_0508.png", "g5_bocpd_noise_by_hour.png", "g5_ablation.png"):
    log(f"  [hình] reports/figures/{p}")


## J. HÌNH
05/08 03–18h (basket SELL sâu): mean P(r ≤ 5) = 0.078, max 0.640; mean p_high = 0.036; tỷ lệ khối argmax = high: 0.028; mean drift = -0.0016
Basket SELL BMSFJA9ZX trong tập chung: 54 dòng; cp TB 0.091; p_high TB 0.095; drift_s TB +0.0037
  [hình] reports/figures/g5_case_0508.png
  [hình] reports/figures/g5_bocpd_noise_by_hour.png
  [hình] reports/figures/g5_ablation.png


In [ ]:
section(log, "K. TIÊU CHÍ HOÀN THÀNH G5 (plan.md)")
crit = [
    ("Dòng bị loại < 1%", pct_all < 1, f"{pct_all:.2f}%"),
    ("k và hazard chốt trước test", True, f"k = {K_MAIN}, 1/{M.LAM_MAIN}"),
    ("Mọi kiểm tra chống nhìn tương lai đạt", all(ok for _, ok, _ in checks), "mục C"),
    ("Họ 4 và họ 5 đều có ΔPR-AUC kèm CI", True, "mục E2"),
    ("Mapping state ghi cho từng fold", len(states) == 3 * len(TEST_MONTHS), "mục D2"),
    ("BOCPD không báo liên tục; HMM ổn định (C3)", not (bocpd_bad or hmm_bad), "mục D4"),
    ("Baseline G1 trên tập đầy đủ = 0,4286", abs(g1_ap - 0.4286) < 5e-5, f"{g1_ap:.4f}"),
]
for name, ok, note in crit:
    log(f"  [{'OK' if ok else '!!'}] {name}  {note}")
out = S[["basket_id", "k", "timestamp", "month"]].copy()
for m in TEST_MONTHS:
    for c in ("cp", "p_high", "drift_s"):
        out[f"{c}_{m}"] = fold_cols[m][c]
out.to_parquet(C.INTERIM / "g5_features.parquet", index=False)
log(f"\nĐã lưu data/interim/g5_features.parquet | tổng thời gian {time.time() - t_start:.0f} s")


## K. TIÊU CHÍ HOÀN THÀNH G5 (plan.md)
  [OK] Dòng bị loại < 1%  0.46%
  [OK] k và hazard chốt trước test  k = 5, 1/120
  [OK] Mọi kiểm tra chống nhìn tương lai đạt  mục C
  [OK] Họ 4 và họ 5 đều có ΔPR-AUC kèm CI  mục E2
  [OK] Mapping state ghi cho từng fold  mục D2
  [OK] BOCPD không báo liên tục; HMM ổn định (C3)  mục D4
  [OK] Baseline G1 trên tập đầy đủ = 0,4286  0.4286

Đã lưu data/interim/g5_features.parquet | tổng thời gian 1189 s
